# Agentic IFRS S1/S2 Report Generation Pipeline — V8.6 expanded report writer

This notebook generates IFRS S1/S2 report sections from synthetic company payloads and the deterministic IFRS requirements KB.

Core design:

- **Content authority:** IFRS requirements + company payload only.
- **Authoring style:** extracted style artifacts under `style_system/authoring`.
- **Missing requirements:** stored in JSON audit files, **not mentioned in the report**.
- **Safety spine:** deterministic claims integrity, factlock, reference firewall, and approval gate.
- **LLM layer:** writer, claims register builder, judges, reviser, and optional fuzzy evidence mapper.
- **PDF layout:** excluded from drafting. `layout_style_guide.json` is used only by the separate PDF assembly stage.

**V8.1 patch:** audit-only paths such as `metadata.data_gaps[*]` are excluded from disclosure plans/writer context, and generic context fields can no longer make a requirement fully `covered` by themselves.


### V8.2 patch — blueprint prose leakage guard

This version keeps the V8.1 writer-safe evidence filtering and additionally sanitizes authoring blueprints before disclosure planning and section writing. Generic blueprint instructions that mention absent/unsupported data, audit-only items, or report-content exclusions are removed so those concepts remain audit/scoring-only and cannot enter report prose.


### V8.6

Adds a controlled expansion/depth layer on top of V8.5. The writer now keeps the same safety rules, but must produce report-like sections with enough narrative depth, entity-specific evidence, and connected explanations. It also adds a deterministic depth gate to prevent safe-but-truncated sections from being approved.


## JSON robustness patch

This version fixes a pipeline crash where the claims-register agent returned malformed or truncated JSON. The JSON parser now saves malformed raw outputs, attempts automatic repair with the strong model, and the claims-register builder has a deterministic fallback so the run can continue to deterministic gates or human review instead of stopping with `JSONDecodeError`.


## V8 strict evidence/scoring patch

Added strict NaN/null/generic-field filtering, improved Strategy routing, missing-requirement audit flags, section-generation scores, and a final-report cleanliness block that prevents missing-data wording from entering the approved report.

In [1]:
# ============================================================
# CELL 1 — SETUP PATHS AND CONFIG
# Notebook expected location: /notebooks
# Style system expected at : /notebooks/gen_data/style/style_system
# ============================================================

import os
from pydoc import resolve
import re
import json
import time
import uuid
import shutil
import random
import urllib.request
import urllib.error
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict, Counter

import pandas as pd

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    raise ImportError("Install python-dotenv first: pip install python-dotenv")

try:
    load_dotenv(find_dotenv(usecwd=True), override=True)
except TypeError:
    load_dotenv(find_dotenv(), override=True)
except AssertionError:
    # Some non-interactive runners cannot inspect call frames for find_dotenv().
    load_dotenv(Path.cwd() / ".env", override=True)

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / "notebooks").exists():
    NOTEBOOK_DIR = (CURRENT_DIR / "notebooks").resolve()
else:
    NOTEBOOK_DIR = CURRENT_DIR

GEN_DATA_DIR = NOTEBOOK_DIR / "gen_data"

# Input folders. Override with env vars if your structure differs.
PAYLOAD_DIR = Path(os.getenv("PAYLOAD_DIR", GEN_DATA_DIR / "payloads")).resolve()
REQUIREMENTS_DIR = Path(os.getenv("IFRS_REQUIREMENTS_DIR", GEN_DATA_DIR / "IFRS" / "ifrs_requirements_kb_outputs_final" / "section_by_section_requirements" / "json")).resolve()
STYLE_SYSTEM_DIR = Path(os.getenv("STYLE_SYSTEM_DIR", GEN_DATA_DIR / "style" / "style_system")).resolve()

# Output folder.
OUTPUT_DIR = Path(os.getenv("GENERATION_OUTPUT_DIR", GEN_DATA_DIR / "generated_reports" / "agentic_ifrs_report")).resolve()

# Pipeline controls.
PIPELINE_MODE = os.getenv("PIPELINE_MODE", "synthetic_demo")
FORBID_INVENTION = True  # hard invariant, not a configurable switch
ALLOW_PARTIAL_COVERAGE = os.getenv("ALLOW_PARTIAL_COVERAGE", "true").lower() == "true"
USE_FUZZY_EVIDENCE_MAPPER = os.getenv("USE_FUZZY_EVIDENCE_MAPPER", "false").lower() == "true"
MAX_REVISION_LOOPS = int(os.getenv("MAX_REVISION_LOOPS", "2"))

# Section order used by the final report.
SECTIONS = [
    "General Requirements",
    "Governance",
    "Strategy",
    "Risk Management",
    "Metrics and Targets",
]

SECTION_SLUGS = {
    "General Requirements": "general_requirements",
    "Governance": "governance",
    "Strategy": "strategy",
    "Risk Management": "risk_management",
    "Metrics and Targets": "metrics_and_targets",
}

# Output subfolders.
DIRS = {
    "evidence_maps": OUTPUT_DIR / "01_evidence_maps",
    "coverage": OUTPUT_DIR / "02_coverage",
    "missing_requirements": OUTPUT_DIR / "03_missing_requirements",
    "plans": OUTPUT_DIR / "04_disclosure_plans",
    "drafts": OUTPUT_DIR / "05_draft_sections",
    "claims": OUTPUT_DIR / "06_claims_registers",
    "gates": OUTPUT_DIR / "07_deterministic_gates",
    "judges": OUTPUT_DIR / "08_judge_results",
    "revisions": OUTPUT_DIR / "09_revised_sections",
    "approved": OUTPUT_DIR / "10_approved_sections",
    "connectivity": OUTPUT_DIR / "11_connectivity",
    "handoff": OUTPUT_DIR / "12_pdf_handoff",
    "audit_logs": OUTPUT_DIR / "audit_logs",
}

for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

print("Current working directory:", CURRENT_DIR)
print("Notebook directory:", NOTEBOOK_DIR)
print("Payload directory:", PAYLOAD_DIR)
print("Requirements directory:", REQUIREMENTS_DIR)
print("Style system directory:", STYLE_SYSTEM_DIR)
print("Output directory:", OUTPUT_DIR)
print("Pipeline mode:", PIPELINE_MODE)
print("Forbid invention:", FORBID_INVENTION)
print("Use fuzzy mapper:", USE_FUZZY_EVIDENCE_MAPPER)

Current working directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks
Notebook directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks
Payload directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\payloads
Requirements directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json
Style system directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system
Output directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report
Pipeline mode: synthetic_demo
Forbid invention: True
Use fuzzy mapper: False


## Azure/OpenAI helper

This cell uses the same full deployment URL style as the working notebook you provided, but keeps this pipeline's model routing:

```env
AZURE_OPENAI_API_KEY=...

AZURE_OPENAI_GPT52_DEPLOYMENT_URL=<full strong GPT-5.2 chat-completions URL>
AZURE_OPENAI_FAST_DEPLOYMENT_URL=<full fast chat-completions URL>
```

The notebook does **not** build or modify endpoint URLs from deployment names. It sends the full URL exactly as configured, after basic quote/markdown cleanup.


In [2]:
# ============================================================
# CELL 2 — LLM CLIENT
# Full deployment URL logic, matching the working REST style.
#
# Uses:
# - AZURE_OPENAI_GPT52_DEPLOYMENT_URL for strong agents
# - AZURE_OPENAI_FAST_DEPLOYMENT_URL for fast/light agents
#
# Important:
# This cell does NOT construct Azure URLs from endpoint + deployment.
# It sends the configured full deployment URL directly.
# ============================================================

import http.client

AZURE_OPENAI_API_KEY = (
    os.getenv("AZURE_OPENAI_API_KEY")
    or os.getenv("OPENAI_API_KEY")
)

# Full Azure / enterprise-gateway chat-completions URLs.
AZURE_OPENAI_GPT52_DEPLOYMENT_URL = os.getenv("AZURE_OPENAI_GPT52_DEPLOYMENT_URL")
AZURE_OPENAI_FAST_DEPLOYMENT_URL = os.getenv("AZURE_OPENAI_FAST_DEPLOYMENT_URL")


def _clean_url(value: Optional[str]) -> Optional[str]:
    """
    Basic cleanup for full deployment URLs.

    Keeps the full URL as provided; does not add/replace api-version.
    Handles:
    - surrounding quotes
    - accidental markdown link format: [label](https://...)
    - accidental copied bracket+url format
    """
    if not value:
        return None

    value = str(value).strip().strip('"').strip("'").strip()

    # Markdown link: [label](https://actual-url)
    md_match = re.search(r"\]\((https://[^)\s]+)\)", value)
    if md_match:
        value = md_match.group(1).strip()

    # Copied format that contains multiple https:// occurrences.
    # Keep the last URL-like occurrence, which is usually the actual href.
    https_positions = [m.start() for m in re.finditer(r"https://", value)]
    if https_positions:
        value = value[https_positions[-1]:]

    value = value.strip().strip("[]").strip()
    value = value.rstrip(").,;")

    return value


AZURE_OPENAI_GPT52_DEPLOYMENT_URL = _clean_url(AZURE_OPENAI_GPT52_DEPLOYMENT_URL)
AZURE_OPENAI_FAST_DEPLOYMENT_URL = _clean_url(AZURE_OPENAI_FAST_DEPLOYMENT_URL)

# Fast deployment falls back to strong if not configured.
if not AZURE_OPENAI_FAST_DEPLOYMENT_URL:
    AZURE_OPENAI_FAST_DEPLOYMENT_URL = AZURE_OPENAI_GPT52_DEPLOYMENT_URL


MODEL_CONFIG = {
    "fuzzy_evidence_mapper": "fast",
    "section_writer": "strong",
    "claims_register_builder": "strong",
    "ifrs_coverage_judge": "strong",
    "evidence_judge": "strong",
    "style_judge": "fast",
    "minimal_reviser": "strong",
    "whole_report_connectivity_judge": "strong",
}


def _mask_url_for_display(url: Optional[str]) -> str:
    """Mask full endpoint URL while keeping enough shape for diagnostics."""
    if not url:
        return "NOT CONFIGURED"

    try:
        import urllib.parse
        parsed = urllib.parse.urlparse(url)

        host = parsed.netloc
        if host:
            host_parts = host.split(".")
            if host_parts and len(host_parts[0]) > 6:
                host_parts[0] = host_parts[0][:3] + "***" + host_parts[0][-2:]
            host = ".".join(host_parts)

        path = parsed.path
        path = re.sub(
            r"(/deployments/)([^/]+)(/chat/completions)",
            lambda m: m.group(1) + m.group(2)[:2] + "***" + m.group(3),
            path,
        )

        # Avoid displaying the raw query because it can make notebook output messy.
        query = "..." if parsed.query else ""

        return urllib.parse.urlunparse((parsed.scheme, host, path, "", query, ""))

    except Exception:
        return "<configured URL, masking failed>"


def validate_llm_config() -> None:
    required = {
        "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
        "AZURE_OPENAI_GPT52_DEPLOYMENT_URL": AZURE_OPENAI_GPT52_DEPLOYMENT_URL,
        "AZURE_OPENAI_FAST_DEPLOYMENT_URL": AZURE_OPENAI_FAST_DEPLOYMENT_URL,
    }

    missing = [name for name, value in required.items() if not value]

    if missing:
        flags = {
            "api_key_loaded": bool(AZURE_OPENAI_API_KEY),
            "gpt52_url_loaded": bool(AZURE_OPENAI_GPT52_DEPLOYMENT_URL),
            "fast_url_loaded": bool(AZURE_OPENAI_FAST_DEPLOYMENT_URL),
        }
        raise ValueError(
            "Missing Azure/OpenAI full-URL configuration values: "
            + ", ".join(missing)
            + "\n\nLoaded flags, keys are never printed:\n"
            + json.dumps(flags, indent=2)
            + "\n\nRequired .env:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_GPT52_DEPLOYMENT_URL=<full GPT-5.2 deployment URL>\n"
              "AZURE_OPENAI_FAST_DEPLOYMENT_URL=<full fast deployment URL>\n"
        )

    for name, url in {
        "AZURE_OPENAI_GPT52_DEPLOYMENT_URL": AZURE_OPENAI_GPT52_DEPLOYMENT_URL,
        "AZURE_OPENAI_FAST_DEPLOYMENT_URL": AZURE_OPENAI_FAST_DEPLOYMENT_URL,
    }.items():
        if not str(url).startswith("https://"):
            raise ValueError(f"{name} must be a full HTTPS deployment URL: {url!r}")

        if "/chat/completions" not in str(url):
            raise ValueError(
                f"{name} does not look like a chat-completions URL.\n"
                f"Configured URL shape: {_mask_url_for_display(url)}\n\n"
                "Expected a full URL ending with /chat/completions plus any required query string."
            )


validate_llm_config()

print("Azure/OpenAI full-URL configuration loaded")
print("Strong endpoint:", _mask_url_for_display(AZURE_OPENAI_GPT52_DEPLOYMENT_URL))
print("Fast endpoint:", _mask_url_for_display(AZURE_OPENAI_FAST_DEPLOYMENT_URL))
print("Model routing:", json.dumps(MODEL_CONFIG, indent=2))


def get_model_url(model_tier: str = "strong") -> str:
    model_tier = (model_tier or "strong").lower().strip()
    if model_tier == "fast":
        return AZURE_OPENAI_FAST_DEPLOYMENT_URL
    return AZURE_OPENAI_GPT52_DEPLOYMENT_URL


def _extract_message_content(data: Dict[str, Any]) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError(
            "Unexpected Azure/OpenAI response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc

    if not content:
        raise ValueError(
            "Azure/OpenAI returned an empty message content. Response:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )

    return content


def _azure_chat_completion(
    *,
    url: str,
    api_key: str,
    messages: List[Dict[str, str]],
    max_output_tokens: int,
    json_mode: bool = False,
    temperature: Optional[float] = None,
    timeout: int = 240,
    request_label: str = "LLM",
    max_attempts: int = 6,
) -> Dict[str, Any]:
    """
    Robust REST call for Azure/OpenAI-compatible enterprise gateways.

    Mirrors the working logic you provided:
    - Uses full deployment URL directly.
    - Retries transient 500/502/503/504 and connection errors.
    - Handles 429 Retry-After.
    - Tries max_completion_tokens first, then max_tokens for gateway compatibility.
    - Does not expose API keys in errors.
    """

    token_fields = ["max_completion_tokens", "max_tokens"]
    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": messages,
            token_field: max_output_tokens,
        }

        if temperature is not None:
            payload["temperature"] = temperature

        if json_mode:
            payload["response_format"] = {"type": "json_object"}

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": api_key,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    return json.loads(resp.read().decode("utf-8"))

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Endpoint: {_mask_url_for_display(url)}\n"
                    f"Token field used: {token_field}\n"
                    f"Response: {body[:3000]}"
                )

                rate_limited = exc.code == 429
                transient = exc.code in {500, 502, 503, 504}
                compatibility_candidate = (
                    token_field == "max_completion_tokens"
                    and exc.code in {400, 422, 500}
                )

                if rate_limited and attempt < max_attempts:
                    retry_after = None
                    try:
                        ra = exc.headers.get("Retry-After") if exc.headers else None
                        if ra is not None:
                            retry_after = float(str(ra).strip())
                    except (TypeError, ValueError):
                        retry_after = None

                    wait = retry_after if retry_after is not None else (2 ** attempt) * 2 + random.random()
                    wait = min(wait, 90)
                    print(
                        f"{request_label}: rate limited (429); retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if transient and attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: server error {exc.code}; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if compatibility_candidate:
                    print(
                        f"{request_label}: gateway may not support "
                        "`max_completion_tokens`; retrying with `max_tokens`."
                    )
                    break

                if exc.code == 404:
                    raise RuntimeError(
                        f"{request_label} HTTP 404 Resource not found.\n"
                        f"Endpoint: {_mask_url_for_display(url)}\n\n"
                        "The notebook is now sending the configured full URL directly. "
                        "So a 404 means the URL itself is not accepted by the gateway, "
                        "or the deployment behind that URL is not accessible with this key.\n\n"
                        "Compare the exact .env value of AZURE_OPENAI_GPT52_DEPLOYMENT_URL "
                        "with the endpoint URL that works in your other notebook."
                    ) from exc

                raise last_error from exc

            except urllib.error.URLError as exc:
                last_error = RuntimeError(
                    f"{request_label} connection error.\n"
                    f"Endpoint: {_mask_url_for_display(url)}\n"
                    f"Error: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection issue; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

            except (ConnectionError, TimeoutError, OSError, http.client.RemoteDisconnected) as exc:
                last_error = RuntimeError(
                    f"{request_label} connection reset/timeout.\n"
                    f"Endpoint: {_mask_url_for_display(url)}\n"
                    f"Error: {type(exc).__name__}: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection reset/timeout "
                        f"({type(exc).__name__}); retrying attempt "
                        f"{attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

    raise last_error or RuntimeError(f"{request_label} request failed for an unknown reason.")



def _strip_markdown_json_fence(text: str) -> str:
    """Remove common ```json fences without touching the JSON body."""
    text = str(text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()


def _extract_balanced_json_object(text: str) -> Optional[str]:
    """
    Return the first balanced JSON object found in text.

    This is safer than taking text[first_brace:last_brace] because model output can
    contain explanatory text, braces inside strings, or multiple JSON-looking blocks.
    If the object is truncated and never balances, return None so the caller can
    attempt LLM repair on the best candidate.
    """
    start = None
    depth = 0
    in_string = False
    escape = False

    for i, ch in enumerate(text):
        if start is None:
            if ch == "{":
                start = i
                depth = 1
            continue

        if escape:
            escape = False
            continue

        if ch == "\\":
            escape = True
            continue

        if ch == '"':
            in_string = not in_string
            continue

        if in_string:
            continue

        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]

    return None


def _extract_json_object(text: str) -> str:
    """
    Extract the most likely JSON object from model output.

    Handles markdown fences and leading/trailing commentary. If the output appears
    truncated, returns the partial object candidate so the repair step can fix it.
    """
    text = _strip_markdown_json_fence(text)

    # Fast path: already a clean JSON object.
    if text.startswith("{") and text.endswith("}"):
        return text

    balanced = _extract_balanced_json_object(text)
    if balanced:
        return balanced

    first = text.find("{")
    last = text.rfind("}")
    if first >= 0 and last > first:
        return text[first:last + 1]
    if first >= 0:
        # Truncated object. Return from first brace onward for repair.
        return text[first:]

    return text


def _json_error_context(candidate: str, exc: json.JSONDecodeError, radius: int = 300) -> str:
    """Small excerpt around a JSONDecodeError location for debugging."""
    pos = getattr(exc, "pos", 0)
    left = max(0, pos - radius)
    right = min(len(candidate), pos + radius)
    excerpt = candidate[left:right]
    pointer = " " * max(0, pos - left) + "^"
    return excerpt + "\n" + pointer


def _safe_debug_filename(label: str) -> str:
    label = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(label or "llm_json"))
    return label.strip("_")[:80] or "llm_json"


def _write_llm_json_debug(raw: str, label: str = "malformed_json") -> Optional[Path]:
    """
    Persist malformed raw LLM output for inspection.
    Uses the notebook audit_logs folder when available.
    """
    try:
        base = DIRS.get("audit_logs", OUTPUT_DIR) if "DIRS" in globals() else Path.cwd()
        base = Path(base) / "llm_json_debug"
        base.mkdir(parents=True, exist_ok=True)
        path = base / f"{time.strftime('%Y%m%d_%H%M%S')}_{_safe_debug_filename(label)}_{uuid.uuid4().hex[:8]}.txt"
        path.write_text(str(raw), encoding="utf-8")
        return path
    except Exception:
        return None


def _repair_json_with_strong_model(malformed_content: str, request_label: str) -> Dict[str, Any]:
    repair_system = (
        "You repair malformed or truncated JSON. "
        "Return one complete valid JSON object only. "
        "Preserve the original meaning, scores, checklist values, issues, and fixes. "
        "Keep strings concise. Do not add markdown fences or commentary."
    )

    repair_user = f"""
Repair the following malformed or truncated output into one complete valid JSON object.

Requirements:
- Keep the same top-level fields when present.
- Finish incomplete strings and arrays conservatively.
- Fix missing commas, unescaped quotes, dangling keys, and truncated arrays.
- Limit each issue/fix/support note string to at most 35 words.
- If the object is a claims register, preserve as many claims as possible but cap at 60 claims.
- Return JSON only.

MALFORMED OUTPUT:
{str(malformed_content)[:70000]}
""".strip()

    data = _azure_chat_completion(
        url=AZURE_OPENAI_GPT52_DEPLOYMENT_URL,
        api_key=AZURE_OPENAI_API_KEY,
        messages=[
            {"role": "system", "content": repair_system},
            {"role": "user", "content": repair_user},
        ],
        max_output_tokens=int(os.getenv("JSON_REPAIR_MAX_TOKENS", "6000")),
        json_mode=True,
        temperature=0,
        request_label=f"{request_label} JSON repair",
    )

    repaired = _extract_message_content(data)
    candidate = _extract_json_object(repaired)
    return json.loads(candidate)


def _parse_or_repair_json(raw: str, request_label: str = "LLM output") -> Dict[str, Any]:
    """
    Parse JSON returned by an LLM. If parsing fails, save the raw output and ask
    the strong model to repair it. This prevents one malformed JSON response from
    crashing the full generation pipeline.
    """
    candidate = _extract_json_object(raw)

    try:
        return json.loads(candidate)
    except json.JSONDecodeError as exc:
        debug_path = _write_llm_json_debug(raw, request_label)
        print(
            f"{request_label}: invalid JSON at line {exc.lineno}, column {exc.colno}. "
            "Attempting JSON repair..."
        )
        if debug_path:
            print("Raw malformed output saved to:", debug_path)
        try:
            return _repair_json_with_strong_model(candidate, request_label=request_label)
        except Exception as repair_exc:
            detail = _json_error_context(candidate, exc)
            raise ValueError(
                f"{request_label}: failed to parse JSON and repair also failed.\n"
                f"Original JSON error: {exc}\n"
                f"Debug file: {debug_path}\n"
                f"Error context:\n{detail}"
            ) from repair_exc

def azure_chat(
    messages: List[Dict[str, str]],
    model_tier: str = "strong",
    temperature: float = 0,
    max_tokens: int = 4000,
    response_format: Optional[Dict[str, str]] = None,
    retries: int = 6,
    retry_sleep: int = 3,
) -> str:
    """
    Azure/OpenAI Chat Completions helper used by all LLM agents.

    Returns text content.
    If response_format={"type": "json_object"}, the model is asked for JSON mode.
    """
    url = get_model_url(model_tier)
    request_label = f"Azure {model_tier} agent"

    json_mode = bool(response_format and response_format.get("type") == "json_object")

    data = _azure_chat_completion(
        url=url,
        api_key=AZURE_OPENAI_API_KEY,
        messages=messages,
        max_output_tokens=max_tokens,
        json_mode=json_mode,
        temperature=temperature,
        request_label=request_label,
        max_attempts=retries,
    )

    return _extract_message_content(data)



def azure_chat_json(
    messages: List[Dict[str, str]],
    model_tier: str = "strong",
    temperature: float = 0,
    max_tokens: int = 4000,
    retries: int = 6,
    request_label: Optional[str] = None,
) -> Dict[str, Any]:
    """
    JSON-safe LLM call.
    First requests JSON mode. If the model returns malformed or truncated JSON,
    parse_json_response repairs it with the strong model.
    """
    label = request_label or f"Azure {model_tier} agent"
    content = azure_chat(
        messages=messages,
        model_tier=model_tier,
        temperature=temperature,
        max_tokens=max_tokens,
        response_format={"type": "json_object"},
        retries=retries,
    )
    return parse_json_response(content, request_label=label)


def parse_json_response(raw: str, request_label: str = "LLM output") -> Dict[str, Any]:
    """
    Backward-compatible parser for cells that call azure_chat(...json mode...).
    Now robust: strict parse first, then automatic repair instead of a hard crash.
    """
    return _parse_or_repair_json(raw, request_label=request_label)


def parse_json_safely(raw: str) -> Dict[str, Any]:
    try:
        return parse_json_response(raw)
    except Exception:
        return {
            "parse_error": True,
            "raw_output_preview": str(raw)[:2000],
        }


def test_llm_connection(model_tier: str = "strong") -> None:
    """Quick smoke test for a configured endpoint."""
    print(f"Testing {model_tier} endpoint:", _mask_url_for_display(get_model_url(model_tier)))

    content = azure_chat(
        [{"role": "user", "content": "Return exactly: OK"}],
        model_tier=model_tier,
        temperature=0,
        max_tokens=20,
    )

    print(f"{model_tier} response:", content)


print("Full-URL LLM helper functions ready")
print("Run test_llm_connection('strong') and test_llm_connection('fast') before running the full pipeline.")


Azure/OpenAI full-URL configuration loaded
Strong endpoint: https://eyq***or.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gp***/chat/completions
Fast endpoint: https://eyq***or.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gp***/chat/completions
Model routing: {
  "fuzzy_evidence_mapper": "fast",
  "section_writer": "strong",
  "claims_register_builder": "strong",
  "ifrs_coverage_judge": "strong",
  "evidence_judge": "strong",
  "style_judge": "fast",
  "minimal_reviser": "strong",
  "whole_report_connectivity_judge": "strong"
}
Full-URL LLM helper functions ready
Run test_llm_connection('strong') and test_llm_connection('fast') before running the full pipeline.


In [3]:
# ============================================================
# CELL 3 — GENERAL UTILITIES
# ============================================================


def slugify(value: str) -> str:
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def read_json(path: Path, default: Any = None) -> Any:
    if not path.exists():
        if default is not None:
            return default
        raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_text(path: Path, default: str = "") -> str:
    if not path.exists():
        return default
    return path.read_text(encoding="utf-8", errors="replace")


def write_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def flatten_json(obj: Any, prefix: str = "") -> Dict[str, Any]:
    """Flatten nested dict/list into path -> scalar/list/dict value."""
    out = {}

    if isinstance(obj, dict):
        for k, v in obj.items():
            new_prefix = f"{prefix}.{k}" if prefix else str(k)
            out.update(flatten_json(v, new_prefix))
    elif isinstance(obj, list):
        if not obj:
            out[prefix] = []
        else:
            for i, v in enumerate(obj):
                new_prefix = f"{prefix}[{i}]"
                out.update(flatten_json(v, new_prefix))
    else:
        out[prefix] = obj

    return out


def get_by_path(obj: Any, path: Any) -> Any:
    """
    Resolve payload paths like a.b[0].c.

    Robustness added:
    - If the LLM returns an evidence source as a dict, try common path keys.
    - If the source is not a string/path-like object, return None instead of crashing.
    - Accept paths copied with a leading "$.".
    """
    if path is None:
        return None

    if isinstance(path, dict):
        for key in ("payload_path", "path", "evidence_path", "source_path", "payloadPath"):
            value = path.get(key)
            if isinstance(value, str) and value.strip():
                path = value
                break
        else:
            return None

    if not isinstance(path, (str, bytes)):
        return None

    path = str(path).strip()
    if not path:
        return None

    if path.startswith("$."):
        path = path[2:]
    elif path.startswith("$"):
        path = path[1:].lstrip(".")

    cur = obj
    tokens = re.findall(r"([^\.\[\]]+)|(\[(\d+)\])", path)
    for name, _, idx in tokens:
        if name:
            if not isinstance(cur, dict) or name not in cur:
                return None
            cur = cur[name]
        elif idx:
            i = int(idx)
            if not isinstance(cur, list) or i >= len(cur):
                return None
            cur = cur[i]
    return cur


def value_preview(value: Any, limit: int = 260) -> str:
    text = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:limit] + ("..." if len(text) > limit else "")


def is_empty_value(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, str) and not value.strip():
        return True
    if isinstance(value, (list, dict)) and len(value) == 0:
        return True
    return False


def tokens(text: str) -> List[str]:
    return [t.lower() for t in re.findall(r"[A-Za-z][A-Za-z0-9_\-]+", str(text)) if len(t) > 2]


def normalize_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "mandatory"}


def now_id() -> str:
    return uuid.uuid4().hex[:10]


## Load inputs

The notebook is tolerant of different file structures. Preferred structure:

```text
/notebooks/gen_data/payloads/
  payload_BANK01_general_requirements.json
  payload_BANK01_governance.json
  payload_BANK01_strategy.json
  payload_BANK01_risk_management.json
  payload_BANK01_metrics_targets.json

/notebooks/gen_data/ifrs_requirements/
  general_requirements_requirements.json
  governance_requirements.json
  strategy_requirements.json
  risk_management_requirements.json
  metrics_and_targets_requirements.json

/notebooks/gen_data/style/style_system/
  authoring/
  judging/
  rendering/
```

In [4]:
# ============================================================
# CELL 4 — LOAD STYLE ARTIFACTS
# ============================================================

AUTHORING_DIR = STYLE_SYSTEM_DIR / "authoring"
JUDGING_DIR = STYLE_SYSTEM_DIR / "judging"
RENDERING_DIR = STYLE_SYSTEM_DIR / "rendering"

# Backward-compatible fallbacks if final organized folders are not present.
if not AUTHORING_DIR.exists():
    AUTHORING_DIR = STYLE_SYSTEM_DIR
if not JUDGING_DIR.exists():
    JUDGING_DIR = STYLE_SYSTEM_DIR / "judging"
if not RENDERING_DIR.exists():
    RENDERING_DIR = STYLE_SYSTEM_DIR / "rendering"

GLOBAL_STYLE = read_json(AUTHORING_DIR / "global_style_guide.json", default={})
STYLE_RUBRIC = read_json(JUDGING_DIR / "style_compliance_rubric.json", default={})

NO_COPYING_RULES = read_text(
    AUTHORING_DIR / "language_rules" / "no_copying_rules.md",
    default=read_text(STYLE_SYSTEM_DIR / "language_rules" / "no_copying_rules.md", default="")
)

TABLE_PATTERNS = read_json(
    AUTHORING_DIR / "table_patterns" / "table_patterns.json",
    default=read_json(STYLE_SYSTEM_DIR / "table_patterns" / "table_patterns.json", default={})
)

FORBIDDEN_TERMS = read_json(
    AUTHORING_DIR / "language_rules" / "forbidden_reference_terms.json",
    default=read_json(STYLE_SYSTEM_DIR / "language_rules" / "forbidden_reference_terms.json", default=[])
)

# Hardcoded safety fallback in case forbidden_reference_terms.json is absent.
FORBIDDEN_TERMS = sorted(set(FORBIDDEN_TERMS + [
    "Emirates NBD", "Emirates NBD Group", "DenizBank", "Emirates Islamic",
    "Dubai", "UAE", "AED", "CBUAE", "Sustainalytics", "KPMG",
    "Microsoft Sustainability Manager"
]))


def load_section_style(section_name: str) -> Dict[str, Any]:
    slug = SECTION_SLUGS[section_name]
    candidates = [
        AUTHORING_DIR / "section_style_guides" / f"{slug}_style.json",
        AUTHORING_DIR / "section_style_guides" / f"{slug}.json",
        STYLE_SYSTEM_DIR / "section_style_guides" / f"{slug}_style.json",
        STYLE_SYSTEM_DIR / "section_style_guides" / f"{slug}.json",
    ]
    for path in candidates:
        if path.exists():
            return read_json(path, default={})
    return {}


def load_section_blueprint(section_name: str) -> Dict[str, Any]:
    slug = SECTION_SLUGS[section_name]
    candidates = [
        AUTHORING_DIR / "section_blueprints" / f"{slug}_blueprint.json",
        AUTHORING_DIR / "section_blueprints" / f"{slug}.json",
        STYLE_SYSTEM_DIR / "section_blueprints" / f"{slug}_blueprint.json",
        STYLE_SYSTEM_DIR / "section_blueprints" / f"{slug}.json",
    ]
    for path in candidates:
        if path.exists():
            return read_json(path, default={})
    return {}

print("Loaded global style:", bool(GLOBAL_STYLE))
print("Loaded table patterns:", bool(TABLE_PATTERNS))
print("Loaded no-copying rules:", bool(NO_COPYING_RULES))
print("Forbidden terms count:", len(FORBIDDEN_TERMS))

Loaded global style: True
Loaded table patterns: True
Loaded no-copying rules: True
Forbidden terms count: 31


In [5]:
# ============================================================
# CELL 5 — LOAD REQUIREMENTS
# V4 PATCH: supports section JSON files nested by standard, e.g.
# {
#   "section_key": "governance",
#   "section_title": "Governance",
#   "row_count": 15,
#   "standards": {
#       "IFRS S1": {"row_count": 7, "requirements": [...]},
#       "IFRS S2": {"row_count": 8, "requirements": [...]}
#   }
# }
# ============================================================

METADATA_KEYS = {
    "section_key",
    "section_title",
    "source",
    "row_count",
    "standards",
    "created_at",
    "metadata",
    "notes",
}


def find_requirements_file(section_name: str) -> Optional[Path]:
    slug = SECTION_SLUGS[section_name]
    candidates = [
        REQUIREMENTS_DIR / f"{slug}_requirements.json",
        REQUIREMENTS_DIR / f"{slug}.json",
        REQUIREMENTS_DIR / "section_by_section_requirements" / "json" / f"{slug}_requirements.json",
        REQUIREMENTS_DIR / "section_by_section_requirements" / "json" / f"{slug}.json",
        REQUIREMENTS_DIR / "section_by_section_requirements" / f"{slug}_requirements.json",
        REQUIREMENTS_DIR / "section_by_section_requirements" / f"{slug}.json",
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


def find_combined_requirements_file() -> Optional[Path]:
    candidates = [
        REQUIREMENTS_DIR / "ifrs_s1_s2_generation_requirements.json",
        REQUIREMENTS_DIR / "generation_requirements.json",
        REQUIREMENTS_DIR / "ifrs_s1_s2_requirements_kb_final.json",
        REQUIREMENTS_DIR / "requirements.json",
        GEN_DATA_DIR / "IFRS" / "ifrs_requirements_kb_outputs_final" / "ifrs_s1_s2_generation_requirements.json",
        GEN_DATA_DIR / "IFRS" / "ifrs_requirements_kb_outputs_final" / "ifrs_s1_s2_requirements_kb_final.json",
        GEN_DATA_DIR / "ifrs_s1_s2_generation_requirements.json",
        GEN_DATA_DIR / "ifrs_s1_s2_requirements_kb_final.json",
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


def _norm_key(value: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def _looks_like_requirement_dict(obj: Dict[str, Any]) -> bool:
    if not isinstance(obj, dict):
        return False
    keys = set(obj.keys())
    return bool(keys.intersection({
        "requirement_id",
        "clean_requirement_text",
        "requirement_text",
        "source_paragraph_text",
        "paragraph_id",
        "report_section",
        "clause_path",
    }))


def _row_has_requirement_text(row: Dict[str, Any]) -> bool:
    text = (
        row.get("clean_requirement_text")
        or row.get("requirement_text")
        or row.get("source_paragraph_text")
        or row.get("text")
        or row.get("paragraph_text")
    )
    return bool(str(text).strip())


def _is_real_requirement_row(row: Any) -> bool:
    if not isinstance(row, dict):
        return False

    rid = str(row.get("requirement_id", "")).strip()
    if not rid or rid in METADATA_KEYS:
        return False

    if not _row_has_requirement_text(row):
        return False

    rid_upper = rid.upper()
    return (
        rid_upper.startswith("IFRS_")
        or "paragraph_id" in row
        or "requirement_text" in row
        or "clean_requirement_text" in row
    )


def rows_from_requirements_object(obj: Any, section_name: str = "") -> List[Dict[str, Any]]:
    """
    Convert many possible JSON shapes into a list of REAL IFRS requirement rows.

    Critical fix:
    Your section requirement files store real rows under:
        obj["standards"]["IFRS S1"]["requirements"]
        obj["standards"]["IFRS S2"]["requirements"]

    The previous notebook iterated over metadata keys such as section_key,
    section_title and source. This function prevents that.
    """
    rows: List[Dict[str, Any]] = []

    # Case 1: already a list of rows.
    if isinstance(obj, list):
        rows = [r for r in obj if isinstance(r, dict)]

    elif isinstance(obj, dict):
        # Case 2: the actual format of your current files.
        if isinstance(obj.get("standards"), dict):
            for standard_name, standard_obj in obj["standards"].items():
                if not isinstance(standard_obj, dict):
                    continue

                reqs = standard_obj.get("requirements", [])
                if isinstance(reqs, dict):
                    reqs = list(reqs.values())

                if isinstance(reqs, list):
                    for req in reqs:
                        if isinstance(req, dict):
                            row = dict(req)
                            row.setdefault("standard", standard_name)
                            row.setdefault("report_section", obj.get("section_title", section_name))
                            rows.append(row)

        # Case 3: common direct list containers.
        elif isinstance(obj.get("requirements"), list):
            rows = [r for r in obj["requirements"] if isinstance(r, dict)]

        elif isinstance(obj.get("generation_requirements"), list):
            rows = [r for r in obj["generation_requirements"] if isinstance(r, dict)]

        elif isinstance(obj.get("items"), list):
            rows = [r for r in obj["items"] if isinstance(r, dict)]

        elif isinstance(obj.get("data"), list):
            rows = [r for r in obj["data"] if isinstance(r, dict)]

        # Case 4: single requirement object.
        elif _looks_like_requirement_dict(obj):
            rows = [obj]

        # Case 5: dict keyed by section names or requirement IDs.
        else:
            section_keys = {
                _norm_key(section_name),
                _norm_key(SECTION_SLUGS.get(section_name, "")),
                _norm_key(section_name.replace("and", "&")),
            }

            # First check whether a value is the matching section container.
            for key, value in obj.items():
                if key in METADATA_KEYS:
                    continue
                if _norm_key(key) in section_keys:
                    rows.extend(rows_from_requirements_object(value, section_name))

            # Otherwise, treat it as dict keyed by requirement_id.
            if not rows:
                for key, value in obj.items():
                    if key in METADATA_KEYS:
                        continue
                    if isinstance(value, dict):
                        row = dict(value)
                        row.setdefault("requirement_id", key)
                        rows.append(row)

    # Final cleanup: keep only real IFRS requirement rows.
    clean_rows = []
    seen = set()
    for row in rows:
        if not _is_real_requirement_row(row):
            continue

        rid = str(row.get("requirement_id", "")).strip()
        if rid in seen:
            continue
        seen.add(rid)
        clean_rows.append(row)

    return clean_rows


def normalize_requirement(row: Any, section_name: str) -> Dict[str, Any]:
    if not isinstance(row, dict):
        raise TypeError(f"Requirement row must be a dict after extraction, got: {type(row).__name__}")

    requirement_text = (
        row.get("clean_requirement_text")
        or row.get("requirement_text")
        or row.get("source_paragraph_text")
        or row.get("text")
        or row.get("paragraph_text")
        or ""
    )

    evidence_tags = row.get("evidence_tags", [])
    if isinstance(evidence_tags, str):
        try:
            parsed = json.loads(evidence_tags)
            evidence_tags = parsed if isinstance(parsed, list) else [parsed]
        except Exception:
            evidence_tags = [x.strip() for x in re.split(r"[,;|]", evidence_tags) if x.strip()]
    elif not isinstance(evidence_tags, list):
        evidence_tags = [str(evidence_tags)] if evidence_tags else []

    return {
        "requirement_id": str(row.get("requirement_id") or row.get("id") or now_id()),
        "standard": row.get("standard", ""),
        "paragraph_id": row.get("paragraph_id", row.get("paragraph", "")),
        "page": row.get("page", ""),
        "report_section": row.get("report_section", section_name),
        "requirement_text": str(requirement_text).strip(),
        "clause_path": row.get("clause_path", ""),
        "obligation_type": row.get("obligation_type", ""),
        "mandatory": normalize_bool(row.get("mandatory", True)),
        "evidence_tags": evidence_tags,
        "banking_relevance": row.get("banking_relevance", ""),
        "raw": row,
    }


def section_matches(row: Dict[str, Any], section_name: str) -> bool:
    sec = str(row.get("report_section", "")).strip()
    if not sec:
        return True
    return _norm_key(sec) == _norm_key(section_name)


def validate_loaded_requirements(section_name: str, rows: List[Dict[str, Any]], source_path: Path) -> None:
    bad_ids = {"section_key", "section_title", "source", "row_count", "standards"}
    ids = {str(r.get("requirement_id", "")) for r in rows}

    if ids.intersection(bad_ids):
        raise ValueError(
            f"Requirement loader bug for {section_name}: metadata keys were loaded as requirements: "
            f"{sorted(ids.intersection(bad_ids))}"
        )

    if not rows:
        raise ValueError(
            f"No real IFRS requirement rows extracted for {section_name} from {source_path}. "
            "Check that the JSON contains standards -> IFRS S1/IFRS S2 -> requirements."
        )


def load_requirements_for_section(section_name: str) -> List[Dict[str, Any]]:
    section_file = find_requirements_file(section_name)

    if section_file:
        obj = read_json(section_file)
        raw_rows = rows_from_requirements_object(obj, section_name)
        rows = [normalize_requirement(r, section_name) for r in raw_rows]
        rows = [r for r in rows if r["requirement_text"] and section_matches(r, section_name)]
        validate_loaded_requirements(section_name, rows, section_file)

        declared_count = obj.get("row_count") if isinstance(obj, dict) else None
        print(f"Loaded requirements for {section_name} from section file: {section_file}")
        print(f"  extracted real IFRS rows: {len(rows)}" + (f" / declared row_count: {declared_count}" if declared_count else ""))
        return rows

    combined_file = find_combined_requirements_file()
    if not combined_file:
        raise FileNotFoundError(
            "Could not find IFRS requirements. Place section JSON files in REQUIREMENTS_DIR "
            "or set IFRS_REQUIREMENTS_DIR in .env."
        )

    obj = read_json(combined_file)
    raw_rows = rows_from_requirements_object(obj, section_name)
    rows = []
    for r in raw_rows:
        nr = normalize_requirement(r, section_name)
        sec = str(nr.get("report_section", "")).strip()
        if _norm_key(sec) == _norm_key(section_name):
            rows.append(nr)

    validate_loaded_requirements(section_name, rows, combined_file)
    print(f"Loaded requirements for {section_name} from combined file: {combined_file}")
    print(f"  extracted real IFRS rows: {len(rows)}")
    return rows


requirements_by_section = {
    section: load_requirements_for_section(section)
    for section in SECTIONS
}

for section, reqs in requirements_by_section.items():
    sample_ids = [r["requirement_id"] for r in reqs[:3]]
    print(f"{section}: {len(reqs)} requirements | sample IDs: {sample_ids}")


Loaded requirements for General Requirements from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json\general_requirements_requirements.json
  extracted real IFRS rows: 108 / declared row_count: 108
Loaded requirements for Governance from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json\governance_requirements.json
  extracted real IFRS rows: 15 / declared row_count: 15
Loaded requirements for Strategy from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json\strategy_requirements.json
  extracted real IFRS rows: 70 / declared row_count: 70
Loaded requirements for Risk Management from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebo

In [6]:
# ============================================================
# CELL 6 — LOAD PAYLOADS
# PATCHED: also searches /notebooks/BANK01 and common project folders.
# ============================================================


def payload_search_dirs() -> List[Path]:
    candidates = [
        PAYLOAD_DIR,
        NOTEBOOK_DIR / "payloads",
        NOTEBOOK_DIR / "data",
        GEN_DATA_DIR / "payloads",
        GEN_DATA_DIR / "BANK01",
        GEN_DATA_DIR / "data",
        CURRENT_DIR / "BANK01",
        CURRENT_DIR / "data",
    ]

    # Keep unique existing-or-configured paths in order.
    out = []
    seen = set()
    for p in candidates:
        p = Path(p).resolve()
        if p not in seen:
            out.append(p)
            seen.add(p)
    return out


def find_payload_file(section_name: str) -> Optional[Path]:
    slug = SECTION_SLUGS[section_name]
    aliases = {
        "metrics_and_targets": ["metrics_targets", "metrics_and_targets", "metrics_targets"],
        "risk_management": ["risk_management", "risk"],
        "general_requirements": ["general_requirements", "general"],
        "governance": ["governance"],
        "strategy": ["strategy"],
    }[slug]

    filename_patterns = []
    for alias in aliases:
        filename_patterns.extend([
            f"payload_BANK01_{alias}.json",
            f"payload_BANK01_{alias}*.json",
            f"BANK01_{alias}.json",
            f"*{alias}*.json",
            f"{alias}.json",
        ])

    for folder in payload_search_dirs():
        if not folder.exists():
            continue
        for pattern in filename_patterns:
            matches = sorted(folder.glob(pattern))
            if matches:
                return matches[0]

    return None


def find_combined_payload_file() -> Optional[Path]:
    filename_patterns = [
        "payload_BANK01.json",
        "payload_BANK01*.json",
        "BANK01.json",
        "payload.json",
        "*payload*.json",
    ]

    for folder in payload_search_dirs():
        if not folder.exists():
            continue
        for pattern in filename_patterns:
            matches = sorted(folder.glob(pattern))
            if matches:
                # Avoid selecting section payload if a combined one exists later.
                section_hint = matches[0].name.lower()
                if any(x in section_hint for x in ["governance", "strategy", "risk_management", "metrics", "general_requirements"]):
                    continue
                return matches[0]

    return None


def load_payload_for_section(section_name: str) -> Dict[str, Any]:
    section_file = find_payload_file(section_name)
    if section_file:
        print(f"Loaded payload for {section_name} from section file: {section_file}")
        return read_json(section_file)

    combined_file = find_combined_payload_file()
    if combined_file:
        print(f"Loaded payload for {section_name} from combined file: {combined_file}")
        combined = read_json(combined_file)
        slug = SECTION_SLUGS[section_name]
        possible_keys = [
            slug,
            slug.replace("metrics_and_targets", "metrics_targets"),
            section_name,
            section_name.lower(),
            section_name.replace(" ", "_").lower(),
        ]
        for key in possible_keys:
            if isinstance(combined, dict) and key in combined:
                return combined[key]
        return combined

    searched = "\n".join([f"- {p}" for p in payload_search_dirs()])
    raise FileNotFoundError(
        "Could not find payload files.\n\n"
        "Searched these folders:\n"
        f"{searched}\n\n"
        "Expected examples:\n"
        "- payload_BANK01_governance.json\n"
        "- payload_BANK01_strategy.json\n"
        "- payload_BANK01_risk_management.json\n"
        "- payload_BANK01_metrics_targets.json\n"
        "- payload_BANK01_general_requirements.json\n"
        "- payload_BANK01.json"
    )


payloads_by_section = {
    section: load_payload_for_section(section)
    for section in SECTIONS
}

for section, payload in payloads_by_section.items():
    flat_count = len(flatten_json(payload))
    print(f"{section}: payload fields={flat_count}")


Loaded payload for General Requirements from combined file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01.json
Loaded payload for Governance from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01_governance.json
Loaded payload for Strategy from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01_strategy.json
Loaded payload for Risk Management from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01_risk_management.json
Loaded payload for Metrics and Targets from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01_metrics_targets.json
General Requirements: payload fields=5122
Governance: payload fields=658
Strategy: payload fields=1098
Risk Management: payload fields=3265
Metrics and Targets: payload fields=1028


## Deterministic evidence mapping and coverage

The evidence mapper is code-first. It maps requirements to actual payload paths. The optional LLM fuzzy mapper can suggest matches, but the path must still resolve to the payload.

Missing requirements are written to JSON audit outputs and are **not passed to the Writer as report content**.

In [7]:
# ============================================================
# CELL 7 — DETERMINISTIC EVIDENCE MAPPER
# V5 PATCH: payload-aware mapping.
#
# Why:
# - Your requirement files are nested by standard and now load correctly.
# - Your payloads are rich section payloads with recurring metadata, bank,
#   reporting_kpis and section-specific tables.
# - Pure lexical matching can over-map broad IFRS requirements to weak fields.
#
# This mapper still remains deterministic/code-first, but adds:
# - section root routing
# - evidence tag routing
# - phrase-to-field boosts
# - metadata noise filtering
# ============================================================

STOPWORDS = {
    "the", "and", "for", "with", "that", "this", "from", "into", "about", "their",
    "shall", "should", "must", "entity", "entities", "information", "disclose",
    "disclosure", "disclosed", "disclosures", "related", "sustainability", "climate",
    "risks", "risk", "opportunities", "opportunity", "reporting", "period",
    "including", "describe", "explain", "enable", "users", "general", "purpose",
    "financial", "reports", "understand", "specific", "specifically", "current",
    "anticipated", "effects", "used", "uses", "use", "accordance", "paragraph",
    "paragraphs", "standard", "standards", "ifrs", "prepare", "preparing",
}

# Section-level allowed roots. These match the top-level tables/objects in your
# BANK01 payload files.
SECTION_ROOTS = {
    "General Requirements": {
        "metadata", "bank", "financial_summary", "general_requirements_context",
        "targets", "scope1", "scope2", "scope3_travel", "financed_emissions",
        "reporting_kpis",
    },
    "Governance": {
        "bank", "governance", "board_minutes", "climate_risk_register",
        "reporting_kpis",
    },
    "Strategy": {
        "metadata", "bank", "financial_summary", "climate_scenarios",
        "climate_risk_register", "value_chain_map", "climate_opportunities",
        "targets", "transition_plan", "resilience_assessment",
        "climate_financial_effects", "reporting_kpis",
    },
    "Risk Management": {
        "metadata", "bank", "climate_risk_register", "physical_risk_exposures",
        "value_chain_map", "governance", "climate_financial_effects",
        "reporting_kpis",
    },
    "Metrics and Targets": {
        "metadata", "bank", "financial_summary", "scope1", "scope2",
        "scope3_travel", "financed_emissions", "financed_emissions_equity",
        "financed_emissions_sovereign", "targets", "carbon_credits",
        "internal_carbon_price", "scope3_categories", "ghg_methodology",
        "scope12_consolidation", "reporting_kpis",
    },
}

# Route IFRS evidence tags to likely payload roots.
TAG_ROOTS = {
    "governance_body": {"governance", "board_minutes"},
    "management_role": {"governance", "board_minutes"},
    "remuneration": {"governance", "board_minutes"},
    "risk_process": {"climate_risk_register", "physical_risk_exposures", "governance", "climate_financial_effects"},
    "scenario_analysis": {"climate_scenarios", "climate_risk_register", "physical_risk_exposures", "resilience_assessment"},
    "business_model_value_chain": {"value_chain_map", "climate_scenarios", "climate_risk_register", "climate_financial_effects"},
    "strategy_decision_making": {"transition_plan", "targets", "climate_opportunities", "climate_scenarios", "climate_risk_register"},
    "financial_effects": {"financial_summary", "climate_financial_effects", "climate_scenarios", "reporting_kpis"},
    "metrics": {
        "scope1", "scope2", "scope3_travel", "financed_emissions",
        "targets", "financial_summary", "reporting_kpis", "ghg_methodology",
        "scope12_consolidation", "scope3_categories", "internal_carbon_price",
        "carbon_credits", "financed_emissions_equity", "financed_emissions_sovereign",
    },
    "targets": {"targets", "reporting_kpis", "governance"},
    "ghg_emissions": {"scope1", "scope2", "scope3_travel", "financed_emissions", "ghg_methodology", "scope12_consolidation", "scope3_categories"},
    "scope_1": {"scope1", "scope12_consolidation"},
    "scope_2": {"scope2", "scope12_consolidation"},
    "scope_3": {"scope3_travel", "scope3_categories", "financed_emissions"},
    "materiality": {"general_requirements_context", "metadata", "climate_risk_register", "value_chain_map"},
    "connected_information": {"general_requirements_context", "financial_summary", "bank", "metadata"},
    "source_guidance": {"general_requirements_context", "metadata", "ghg_methodology"},
}

# Phrase rules connect common IFRS wording to expected payload roots and fields.
# Each rule: (phrases in requirement text, preferred roots, path/value hints)
PHRASE_RULES = [
    (["governance body", "body", "board", "committee", "charged with governance"], ["governance", "board_minutes"], ["board", "committee", "governance", "members_present", "meeting"]),
    (["skills", "competencies", "competence"], ["governance"], ["skill", "expertise", "training", "development", "competenc"]),
    (["how often", "informed"], ["governance", "board_minutes"], ["frequency", "meeting", "minutes", "agenda", "reporting_to_board", "climate_risk_reporting"]),
    (["major transactions", "trade-offs", "trade offs"], ["governance", "board_minutes", "transition_plan"], ["major_transactions", "decision", "trade", "transition_plan"]),
    (["targets", "progress"], ["targets", "governance", "reporting_kpis"], ["target", "progress", "baseline", "remuneration"]),
    (["remuneration", "compensation"], ["governance"], ["compensation", "remuneration", "ceo", "exec"]),
    (["management", "controls", "procedures"], ["governance", "climate_risk_register"], ["management", "committee", "erm", "control", "integrated"]),
    (["identify", "assess", "prioritise", "prioritize", "monitor"], ["climate_risk_register", "physical_risk_exposures"], ["risk_rating", "likelihood", "severity", "monitoring_frequency", "risk_name", "risk_category", "risk_description", "high_risk_flag"]),
    (["scenario analysis"], ["climate_scenarios", "climate_risk_register", "resilience_assessment"], ["scenario", "scenario_analysis", "framework", "horizon", "methodology", "resilience"]),
    (["changed", "previous reporting period"], ["climate_risk_register"], ["changed_since_prior_period"]),
    (["integrated", "overall risk management"], ["climate_risk_register", "governance"], ["erm_integrated", "erm_integration", "management"]),
    (["business model", "value chain"], ["value_chain_map"], ["value_chain", "node", "upstream", "downstream", "business_model"]),
    (["financial position", "financial performance", "cash flows", "financial effects"], ["climate_financial_effects", "financial_summary"], ["affected_statement", "line_item", "quantitative_effect", "financial", "cash", "performance", "revenue", "profit"]),
    (["resilience", "climate resilience"], ["resilience_assessment", "climate_scenarios"], ["resilience", "capacity", "scenario", "uncertainties"]),
    (["transition plan"], ["transition_plan", "climate_scenarios", "targets"], ["transition_plan", "net_zero", "dependencies", "resourcing", "assumptions"]),
    (["greenhouse gas", "ghg", "emissions", "co2"], ["scope1", "scope2", "scope3_travel", "financed_emissions", "ghg_methodology"], ["scope", "emissions", "tco2e", "ghg"]),
    (["scope 1"], ["scope1", "scope12_consolidation"], ["scope1"]),
    (["scope 2"], ["scope2", "scope12_consolidation"], ["scope2", "market", "location"]),
    (["scope 3", "financed emissions"], ["scope3_travel", "financed_emissions", "scope3_categories", "financed_emissions_equity", "financed_emissions_sovereign"], ["scope3", "financed", "category", "attributed"]),
    (["carbon price", "internal carbon"], ["internal_carbon_price", "climate_scenarios"], ["carbon_price", "internal_carbon"]),
    (["capital deployment", "capital expenditure", "financing", "investment deployed"], ["financial_summary", "reporting_kpis"], ["capex", "opex", "climate_capex", "investment", "financing"]),
    (["comparative", "revised comparative", "redefines", "replaces", "estimate"], ["financial_summary", "scope1", "scope2", "financed_emissions", "metadata"], ["2022", "2023", "comparative", "estimate", "data_gaps"]),
    (["data source", "inputs", "parameters", "measurement approach", "method"], ["metadata", "ghg_methodology", "scope12_consolidation", "climate_scenarios", "physical_risk_exposures"], ["method", "source", "data_source", "input", "assumption", "basis", "scope"]),
    (["reporting entity", "same reporting entity", "financial statements", "currency", "reporting period"], ["bank", "general_requirements_context", "financial_summary"], ["reporting", "currency", "entity", "period", "fiscal", "boundary"]),
    (["material"], ["general_requirements_context", "metadata", "climate_risk_register"], ["materiality", "material", "risk_rating", "high_risk"]),
]

NOISE_PATH_FRAGMENTS = [
    "coherence_fixes_applied",
]


def root_of_path(path: str) -> str:
    return re.split(r"[.\[]", str(path), maxsplit=1)[0]


def requirement_text_blob(req: Dict[str, Any]) -> str:
    tags = req.get("evidence_tags", [])
    if isinstance(tags, str):
        tags = [tags]
    return " ".join([
        str(req.get("requirement_text", "")),
        str(req.get("clause_path", "")),
        " ".join([str(t).replace("_", " ") for t in tags]),
    ]).lower()


def allowed_roots_for_requirement(section_name: str, req: Dict[str, Any]) -> set:
    roots = set(SECTION_ROOTS.get(section_name, set()))

    tags = req.get("evidence_tags", [])
    if isinstance(tags, str):
        tags = [tags]

    for tag in tags:
        roots |= TAG_ROOTS.get(str(tag), set())

    text = requirement_text_blob(req)
    for phrases, preferred_roots, _path_hints in PHRASE_RULES:
        if any(phrase in text for phrase in phrases):
            roots |= set(preferred_roots)

    return roots


def requirement_keywords(req: Dict[str, Any]) -> List[str]:
    parts = [
        req.get("requirement_text", ""),
        req.get("clause_path", ""),
        req.get("obligation_type", ""),
        req.get("banking_relevance", ""),
    ]

    tags = req.get("evidence_tags", [])
    if isinstance(tags, str):
        try:
            tags = json.loads(tags)
        except Exception:
            tags = re.split(r"[,;|]", tags)
    elif not isinstance(tags, list):
        tags = [tags] if tags else []

    parts.extend([str(t).replace("_", " ") for t in tags])

    kws = []
    for part in parts:
        kws.extend(tokens(str(part).replace("_", " ")))

    return sorted(set([k for k in kws if k not in STOPWORDS and len(k) > 2]))


def field_keywords(path: str, value: Any) -> List[str]:
    text = str(path).replace("_", " ").replace(".", " ")
    if isinstance(value, (str, int, float, bool)):
        text += " " + str(value).replace("_", " ")
    elif isinstance(value, (dict, list)):
        text += " " + value_preview(value, limit=500).replace("_", " ")
    return [t for t in tokens(text) if t not in STOPWORDS and len(t) > 2]


def phrase_path_boost(req: Dict[str, Any], path: str, value: Any) -> Tuple[int, List[str]]:
    text = requirement_text_blob(req)
    path_text = str(path).lower().replace("_", " ")
    value_text = str(value).lower().replace("_", " ") if isinstance(value, (str, int, float, bool)) else ""
    root = root_of_path(path)

    total = 0
    hits = []

    for phrases, preferred_roots, path_hints in PHRASE_RULES:
        if not any(phrase in text for phrase in phrases):
            continue

        matched_hints = [
            hint for hint in path_hints
            if hint.replace("_", " ") in path_text or hint.replace("_", " ") in value_text
        ]

        if matched_hints:
            total += 4
            hits.extend(matched_hints[:4])
        elif root in preferred_roots:
            total += 2

    return total, sorted(set(hits))


def metadata_allowed_for_requirement(req: Dict[str, Any], path: str) -> bool:
    """
    Metadata is valuable for data gaps, methodology, reporting basis and assumptions,
    but should not dominate every requirement.
    """
    text = requirement_text_blob(req)
    relevant_terms = [
        "data", "comparative", "estimate", "measurement", "method", "source",
        "unavailable", "gap", "limitation", "scope", "basis", "assumption",
        "currency", "period", "reporting entity", "financial statements",
        "guidance",
    ]

    if "metadata.data_gaps" in path:
        return any(term in text for term in relevant_terms)

    if "metadata.pcaf_methodology" in path:
        return any(term in text for term in ["method", "source", "emission", "financed", "scope 3", "data", "estimate"])

    return True


def evidence_score(req: Dict[str, Any], section_name: str, path: str, value: Any) -> Tuple[int, List[str], str]:
    root = root_of_path(path)

    if any(fragment in str(path) for fragment in NOISE_PATH_FRAGMENTS):
        return -999, [], "excluded_noise_path"

    if root == "metadata" and not metadata_allowed_for_requirement(req, str(path)):
        return -999, [], "metadata_not_relevant_to_requirement"

    allowed_roots = allowed_roots_for_requirement(section_name, req)

    req_kws = set(requirement_keywords(req))
    f_kws = set(field_keywords(path, value))
    overlap = sorted(req_kws.intersection(f_kws))

    score = len(overlap)

    path_lower = str(path).lower()
    for kw in req_kws:
        if len(kw) > 3 and kw in path_lower:
            score += 1

    route_reason = "lexical"

    if allowed_roots:
        if root in allowed_roots:
            score += 3
            route_reason = "payload_root_routing+lexical"
        else:
            score -= 3
            route_reason = "outside_expected_payload_root"

    boost, phrase_hits = phrase_path_boost(req, path, value)
    if boost:
        score += boost
        route_reason = "payload_root_routing+phrase_boost+lexical"

    # Mild recency/context boost; never sufficient alone.
    if "reporting_year" in path_lower or "2024" in str(value):
        score += 1

    return score, sorted(set(overlap + phrase_hits)), route_reason


def build_evidence_map_for_section(
    section_name: str,
    requirements: List[Dict[str, Any]],
    payload: Dict[str, Any],
    top_k: int = 8,
    min_score: int = 5,
) -> List[Dict[str, Any]]:
    flat = flatten_json(payload)
    non_empty_items = [(p, v) for p, v in flat.items() if not is_empty_value(v)]
    mapped = []

    for req in requirements:
        candidates = []
        for path, value in non_empty_items:
            score, matched_terms, reason = evidence_score(req, section_name, path, value)

            if score >= min_score:
                candidates.append({
                    "payload_path": path,
                    "payload_root": root_of_path(path),
                    "value_preview": value_preview(value),
                    "value_type": type(value).__name__,
                    "match_score": score,
                    "matched_keywords": matched_terms,
                    "mapping_reason": reason,
                })

        candidates = sorted(
            candidates,
            key=lambda x: (x["match_score"], len(x.get("matched_keywords", []))),
            reverse=True,
        )[:top_k]

        mapped.append({
            "requirement_id": req["requirement_id"],
            "section_name": section_name,
            "requirement_text": req["requirement_text"],
            "mandatory": req["mandatory"],
            "evidence_candidates": candidates,
            "mapping_method": "deterministic_payload_aware",
        })

    return mapped


def summarize_evidence_map(section_name: str, evidence_map: List[Dict[str, Any]]) -> Dict[str, Any]:
    roots = Counter()
    candidate_counts = []
    for row in evidence_map:
        candidate_counts.append(len(row.get("evidence_candidates", [])))
        for c in row.get("evidence_candidates", []):
            roots[c.get("payload_root") or root_of_path(c.get("payload_path", ""))] += 1

    covered = sum(1 for x in candidate_counts if x > 0)
    return {
        "section_name": section_name,
        "requirements_total": len(evidence_map),
        "requirements_with_candidates": covered,
        "requirements_without_candidates": len(evidence_map) - covered,
        "candidate_root_distribution": dict(roots.most_common()),
    }



# V8 performance cache: prevents recomputing requirement/path tokens for every candidate pair.
_REQUIREMENT_TEXT_BLOB_CACHE = {}
_REQUIREMENT_KEYWORDS_CACHE = {}
_FIELD_KEYWORDS_CACHE = {}


def requirement_text_blob(req: Dict[str, Any]) -> str:  # noqa: F811 - intentional cached override
    rid = str(req.get("requirement_id", id(req)))
    if rid in _REQUIREMENT_TEXT_BLOB_CACHE:
        return _REQUIREMENT_TEXT_BLOB_CACHE[rid]
    tags = req.get("evidence_tags", [])
    if isinstance(tags, str):
        tags = [tags]
    text = " ".join([
        str(req.get("requirement_text", "")),
        str(req.get("clause_path", "")),
        " ".join([str(t).replace("_", " ") for t in tags]),
    ]).lower()
    _REQUIREMENT_TEXT_BLOB_CACHE[rid] = text
    return text


def requirement_keywords(req: Dict[str, Any]) -> List[str]:  # noqa: F811 - intentional cached override
    rid = str(req.get("requirement_id", id(req)))
    if rid in _REQUIREMENT_KEYWORDS_CACHE:
        return _REQUIREMENT_KEYWORDS_CACHE[rid]
    parts = [
        req.get("requirement_text", ""),
        req.get("clause_path", ""),
        req.get("obligation_type", ""),
        req.get("banking_relevance", ""),
    ]
    tags = req.get("evidence_tags", [])
    if isinstance(tags, str):
        try:
            tags = json.loads(tags)
        except Exception:
            tags = re.split(r"[,;|]", tags)
    elif not isinstance(tags, list):
        tags = [tags] if tags else []
    parts.extend([str(t).replace("_", " ") for t in tags])
    kws = []
    for part in parts:
        kws.extend(tokens(str(part).replace("_", " ")))
    result = sorted(set([k for k in kws if k not in STOPWORDS and len(k) > 2]))
    _REQUIREMENT_KEYWORDS_CACHE[rid] = result
    return result


def field_keywords(path: str, value: Any) -> List[str]:  # noqa: F811 - intentional cached override
    # Path is unique inside a flattened payload. Include a short value preview to avoid collisions across sections.
    key = (str(path), type(value).__name__, value_preview(value, limit=200) if not isinstance(value, (int, float, bool)) else str(value))
    if key in _FIELD_KEYWORDS_CACHE:
        return _FIELD_KEYWORDS_CACHE[key]
    text = str(path).replace("_", " ").replace(".", " ")
    if isinstance(value, (str, int, float, bool)):
        text += " " + str(value).replace("_", " ")
    elif isinstance(value, (dict, list)):
        text += " " + value_preview(value, limit=500).replace("_", " ")
    result = [t for t in tokens(text) if t not in STOPWORDS and len(t) > 2]
    _FIELD_KEYWORDS_CACHE[key] = result
    return result


# Build and save evidence maps.
evidence_maps_by_section = {}
evidence_map_summaries = {}

for section in SECTIONS:
    evidence_map = build_evidence_map_for_section(
        section,
        requirements_by_section[section],
        payloads_by_section[section],
    )
    evidence_maps_by_section[section] = evidence_map

    summary = summarize_evidence_map(section, evidence_map)
    evidence_map_summaries[section] = summary

    slug = SECTION_SLUGS[section]
    path = DIRS["evidence_maps"] / f"evidence_map_{slug}.json"
    write_json(evidence_map, path)
    write_json(summary, DIRS["evidence_maps"] / f"evidence_map_summary_{slug}.json")

    print(
        section,
        "mapped", len(evidence_map), "requirements |",
        "with candidates:", summary["requirements_with_candidates"],
        "| without:", summary["requirements_without_candidates"],
        "->", path
    )

General Requirements mapped 108 requirements | with candidates: 94 | without: 14 -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_general_requirements.json
Governance mapped 15 requirements | with candidates: 15 | without: 0 -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_governance.json
Strategy mapped 70 requirements | with candidates: 66 | without: 4 -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_strategy.json
Risk Management mapped 17 requirements | with candidates: 17 | without: 0 -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_risk_management.json
Metrics and Targets mapped 151 requirements | with candidate

In [8]:
# ============================================================
# CELL 8 — OPTIONAL FUZZY EVIDENCE MAPPER LLM FALLBACK
# Use only for unresolved requirements. The path still must exist.
# ============================================================


def fuzzy_map_unresolved_requirements(
    section_name: str,
    requirements: List[Dict[str, Any]],
    payload: Dict[str, Any],
    evidence_map: List[Dict[str, Any]],
    max_unresolved: int = 20,
) -> List[Dict[str, Any]]:
    unresolved = [m for m in evidence_map if not m["evidence_candidates"]]
    if not unresolved or not USE_FUZZY_EVIDENCE_MAPPER:
        return evidence_map

    unresolved = unresolved[:max_unresolved]
    flat = flatten_json(payload)
    payload_catalog = [
        {"payload_path": p, "value_preview": value_preview(v, 120)}
        for p, v in flat.items()
        if not is_empty_value(v)
    ][:300]

    prompt = {
        "section_name": section_name,
        "task": "Suggest payload paths that may support unresolved IFRS requirements. Only use paths from payload_catalog.",
        "rules": [
            "Do not invent payload paths.",
            "Return an empty list if no path supports a requirement.",
            "A suggested path must be semantically relevant, not merely same section.",
        ],
        "unresolved_requirements": [
            {
                "requirement_id": m["requirement_id"],
                "requirement_text": m["requirement_text"],
                "mandatory": m["mandatory"],
            }
            for m in unresolved
        ],
        "payload_catalog": payload_catalog,
    }

    messages = [
        {"role": "system", "content": "You are a precise evidence mapping assistant. Return JSON only."},
        {"role": "user", "content": json.dumps(prompt, ensure_ascii=False)},
    ]
    raw = azure_chat(
        messages,
        model_tier=MODEL_CONFIG["fuzzy_evidence_mapper"],
        temperature=0,
        max_tokens=3000,
        response_format={"type": "json_object"},
    )
    obj = parse_json_response(raw, request_label=f"fuzzy_evidence_mapper_{SECTION_SLUGS[section_name]}")
    suggestions = obj.get("suggestions", [])

    existing_paths = set(flat.keys())
    by_req = defaultdict(list)
    for s in suggestions:
        rid = s.get("requirement_id")
        for path in s.get("payload_paths", []):
            if path in existing_paths:
                by_req[rid].append({
                    "payload_path": path,
                    "value_preview": value_preview(flat[path]),
                    "value_type": type(flat[path]).__name__,
                    "match_score": int(s.get("confidence", 1)),
                    "matched_keywords": ["llm_fuzzy_match"],
                    "llm_reason": s.get("reason", ""),
                })

    for m in evidence_map:
        if not m["evidence_candidates"] and m["requirement_id"] in by_req:
            m["evidence_candidates"] = by_req[m["requirement_id"]]
            m["mapping_method"] = "llm_fuzzy_verified_path"

    return evidence_map

if USE_FUZZY_EVIDENCE_MAPPER:
    for section in SECTIONS:
        updated = fuzzy_map_unresolved_requirements(
            section,
            requirements_by_section[section],
            payloads_by_section[section],
            evidence_maps_by_section[section],
        )
        evidence_maps_by_section[section] = updated
        write_json(updated, DIRS["evidence_maps"] / f"evidence_map_{SECTION_SLUGS[section]}.json")
        print("Fuzzy mapping completed for", section)
else:
    print("Fuzzy evidence mapper disabled.")

Fuzzy evidence mapper disabled.


In [9]:
# ============================================================
# CELL 9 — COVERAGE CLASSIFIER + MISSING REQUIREMENTS REGISTER
# Missing requirements are NOT report content. They are audit JSON.
# V5 PATCH: use payload-aware mapping confidence.
# ============================================================


def classify_requirement_coverage(req: Dict[str, Any], mapping: Dict[str, Any]) -> Dict[str, Any]:
    candidates = mapping.get("evidence_candidates", [])
    mandatory = req.get("mandatory", True)

    if candidates:
        max_score = max([c.get("match_score", 0) for c in candidates] or [0])

        # Payload-aware mapper uses min_score=5.
        # covered      : strong route + phrase/field support
        # partial      : some evidence exists but may not satisfy full clause
        if max_score >= 8:
            status = "covered"
        elif max_score >= 5:
            status = "partially_covered"
        else:
            status = "not_available_in_payload" if mandatory else "not_applicable"
    else:
        if mandatory:
            status = "not_available_in_payload"
        else:
            raw_text = json.dumps(req.get("raw", {}), ensure_ascii=False).lower()
            if any(x in raw_text for x in ["if applicable", "when applicable", "where applicable", "conditional"]):
                status = "not_applicable"
            else:
                status = "not_available_in_payload"

    return {
        "requirement_id": req["requirement_id"],
        "standard": req.get("standard", ""),
        "paragraph_id": req.get("paragraph_id", ""),
        "report_section": req.get("report_section", ""),
        "requirement_text": req.get("requirement_text", ""),
        "mandatory": mandatory,
        "coverage_status": status,
        "evidence_count": len(candidates),
        "evidence_confidence_score": max([c.get("match_score", 0) for c in candidates] or [0]),
        "evidence_paths": [c["payload_path"] for c in candidates],
        "not_applicable_justification": "Conditional/non-mandatory requirement with no relevant synthetic payload evidence." if status == "not_applicable" else "",
    }


def build_coverage_and_missing_register(section_name: str) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    reqs = requirements_by_section[section_name]
    maps = {m["requirement_id"]: m for m in evidence_maps_by_section[section_name]}

    coverage = []
    missing = []

    for req in reqs:
        cov = classify_requirement_coverage(req, maps[req["requirement_id"]])
        coverage.append(cov)

        if cov["coverage_status"] == "not_available_in_payload":
            missing.append({
                "requirement_id": cov["requirement_id"],
                "standard": cov["standard"],
                "paragraph_id": cov["paragraph_id"],
                "report_section": section_name,
                "mandatory": cov["mandatory"],
                "requirement_text": cov["requirement_text"],
                "coverage_status": "not_available_in_payload",
                "reason": "No sufficiently relevant payload evidence was identified for this requirement.",
                "action_needed": "Add evidence for this requirement to the section payload or map an existing payload field manually.",
                "report_instruction": "Do not mention this missing requirement in the generated report.",
            })

    missing_register = {
        "section_name": section_name,
        "pipeline_mode": PIPELINE_MODE,
        "policy": "Missing requirements are recorded here and are excluded from report prose.",
        "missing_requirements": missing,
    }
    return coverage, missing_register


coverage_by_section = {}
missing_registers_by_section = {}

for section in SECTIONS:
    coverage, missing_register = build_coverage_and_missing_register(section)
    coverage_by_section[section] = coverage
    missing_registers_by_section[section] = missing_register

    slug = SECTION_SLUGS[section]
    write_json(coverage, DIRS["coverage"] / f"coverage_matrix_{slug}.json")
    write_json(missing_register, DIRS["missing_requirements"] / f"missing_requirements_{slug}.json")

    counts = Counter([c["coverage_status"] for c in coverage])
    print(section, dict(counts), "missing:", len(missing_register["missing_requirements"]))

combined_missing = {
    "pipeline_mode": PIPELINE_MODE,
    "policy": "The report contains only evidence-supported disclosures. Missing requirements are stored here, not in the report.",
    "sections": missing_registers_by_section,
}
write_json(combined_missing, DIRS["missing_requirements"] / "missing_requirements_all_sections.json")

General Requirements {'covered': 79, 'not_available_in_payload': 14, 'partially_covered': 15} missing: 14
Governance {'covered': 15} missing: 0
Strategy {'not_available_in_payload': 4, 'covered': 57, 'partially_covered': 9} missing: 4
Risk Management {'covered': 17} missing: 0
Metrics and Targets {'covered': 142, 'partially_covered': 7, 'not_available_in_payload': 2} missing: 2


## V8 strict evidence, missing-requirement flagging, and section scoring patch

This patch keeps missing requirements out of report prose. Missing requirements are written only to audit JSON/Markdown outputs and are used for scoring and human review decisions.

In [10]:
# ============================================================
# CELL 9B — STRICT EVIDENCE, COVERAGE, MISSING FLAGS + SCORING PATCH V8.1
# ============================================================
# Fast post-processor over CELL 7 maps:
# - removes null/NaN/generic evidence candidates
# - adds targeted Strategy/General routes for known high-level clauses
# - recalculates coverage using strong vs medium evidence
# - writes missing-requirement flags as audit-only outputs
# ============================================================

MISSING_LIKE_STRINGS = {
    "", "nan", "none", "null", "na", "n/a", "not applicable", "not_applicable"
}

GENERIC_CONTEXT_LEAVES = {
    "reporting_year", "bank_id", "summary_id", "id", "country", "lei_code",
    "fiscal_year_end", "boundary_type", "reporting_currency", "established_year",
    "headcount", "in_scope_esg_flag", "regulatory_regime"
}

GENERIC_ALLOWED_TERMS = {
    "reporting period", "reporting year", "same reporting", "reporting entity",
    "financial statements", "presentation currency", "currency", "fiscal",
    "comparative", "preceding period", "prior period", "boundary", "general purpose financial reports",
    "same time", "period covered", "longer or shorter than 12 months"
}


# V8.1 PATCH — audit-only evidence may help scoring/flagging but must never be
# passed to section writers or used as strong support for "covered".
AUDIT_ONLY_PATH_FRAGMENTS = {
    "data_gaps",
    "data_gap",
    "missing_requirement",
    "missing_requirements",
    "not_available",
    "unavailable",
}

AUDIT_ONLY_LEAVES = {
    "sovereign_bonds_with_data_gaps",
    "listed_equity_emissions_are_proxy",
}

def is_audit_only_evidence_path(path: str) -> bool:
    p = str(path).lower()
    leaf = path_leaf(path) if "path_leaf" in globals() else re.split(r"[.\[\]]+", p)[-1]
    return any(fragment in p for fragment in AUDIT_ONLY_PATH_FRAGMENTS) or leaf in AUDIT_ONLY_LEAVES

def writer_evidence_path_allowed(path: str) -> bool:
    """Evidence safety gate for disclosure plans and LLM writer context."""
    return not is_audit_only_evidence_path(path)

STRICT_EXTRA_PHRASE_RULES = [
    (["risks and opportunities that could reasonably be expected", "risks and opportunities", "affect the entity's prospects", "affect the entity’s prospects"],
     ["climate_risk_register", "climate_opportunities", "value_chain_map"],
     ["risk_name", "risk_description", "risk_category", "opportunity", "description", "time_horizon", "materiality"]),
    (["strategy and decision-making", "strategy and decision making", "responded to", "plans to respond", "strategic response"],
     ["transition_plan", "climate_scenarios", "climate_opportunities", "targets", "climate_financial_effects"],
     ["transition_plan", "resilience", "scenario", "target", "progress", "opportunity", "financial_effect", "mitigation"]),
    (["fair presentation", "complete, neutral and accurate", "faithful representation", "statement of compliance", "apply this standard"],
     ["general_requirements_context", "metadata", "bank"],
     ["standards_basis", "assurance", "reporting", "regulatory_regime", "source_systems"]),
    (["judgements", "approximations", "assumptions", "measurement uncertainty", "sources of measurement uncertainty"],
     ["general_requirements_context", "metadata", "ghg_methodology", "scope12_consolidation", "financial_summary", "reporting_kpis"],
     ["methodology", "assumption", "estimate", "data_gaps", "quality", "source", "pcaf", "scope2_rec_reconciliation"]),
]

_existing_rule_keys = {tuple(r[0]) for r in PHRASE_RULES}
for _rule in reversed(STRICT_EXTRA_PHRASE_RULES):
    if tuple(_rule[0]) not in _existing_rule_keys:
        PHRASE_RULES.insert(0, _rule)

TAG_ROOTS.update({
    "reporting_basis": {"general_requirements_context", "metadata", "bank"},
    "compliance_basis": {"general_requirements_context", "metadata", "bank"},
    "measurement_uncertainty": {"general_requirements_context", "metadata", "ghg_methodology", "scope12_consolidation", "reporting_kpis"},
    "strategy_response": {"transition_plan", "climate_scenarios", "climate_opportunities", "targets", "climate_financial_effects"},
    "risks_opportunities": {"climate_risk_register", "climate_opportunities", "value_chain_map"},
})

REQUIREMENT_ID_ROUTE_HINTS = {
    "IFRS_S1_29_C01": ({"climate_risk_register", "climate_opportunities"}, {"risk_name", "risk_description", "risk_category", "description", "opportunity_name", "time_horizon"}),
    "IFRS_S1_30_C01": ({"climate_risk_register", "climate_opportunities"}, {"risk_name", "risk_description", "risk_category", "description", "opportunity_name", "time_horizon"}),
    "IFRS_S2_9_C01": ({"climate_risk_register", "climate_opportunities", "climate_scenarios"}, {"risk_name", "risk_description", "risk_category", "scenario", "description", "time_horizon"}),
    "IFRS_S2_10_C01": ({"climate_risk_register", "climate_opportunities", "climate_scenarios"}, {"risk_name", "risk_description", "risk_category", "scenario", "description", "time_horizon"}),
    "IFRS_S2_10_C02": ({"climate_risk_register"}, {"risk_category", "risk_name", "risk_description", "physical", "transition"}),
    "IFRS_S1_29_C03": ({"transition_plan", "climate_scenarios", "climate_opportunities", "targets", "climate_financial_effects"}, {"transition_plan", "resilience", "methodology_notes", "target", "progress", "opportunity", "mitigation_actions"}),
    "IFRS_S1_33_C01": ({"transition_plan", "climate_scenarios", "climate_opportunities", "targets", "climate_financial_effects"}, {"transition_plan", "resilience", "methodology_notes", "target", "progress", "opportunity", "mitigation_actions"}),
    "IFRS_S2_9_C03": ({"transition_plan", "climate_scenarios", "targets", "climate_financial_effects"}, {"transition_plan", "resilience", "methodology_notes", "target", "progress"}),
    "IFRS_S1_5_C01": ({"general_requirements_context", "metadata", "bank"}, {"standards_basis", "regulatory_regime", "source_systems"}),
    "IFRS_S1_11_C01": ({"general_requirements_context", "metadata"}, {"standards_basis", "source_systems", "assurance", "risk_rating_methodology"}),
    "IFRS_S1_13_C01": ({"general_requirements_context", "metadata"}, {"standards_basis", "source_systems", "assurance", "risk_rating_methodology"}),
    "IFRS_S1_15_C01": ({"general_requirements_context", "metadata", "reporting_kpis"}, {"assurance", "source_systems", "emissions_data_quality", "data_quality"}),
    "IFRS_S1_21_C01": ({"general_requirements_context", "metadata", "financial_summary", "reporting_kpis"}, {"source_systems", "reporting", "financial", "data_gaps"}),
    "IFRS_S1_B39_C01": ({"general_requirements_context", "metadata", "financial_summary", "reporting_kpis"}, {"source_systems", "reporting", "financial", "data_gaps"}),
    "IFRS_S1_25_C02": ({"climate_scenarios", "transition_plan", "climate_opportunities", "targets", "financial_summary"}, {"transition_plan", "resilience", "scenario", "target", "opportunity", "climate_capex"}),
}


def path_leaf(path: str) -> str:
    parts = re.split(r"[.\[\]]+", str(path))
    return next((p for p in reversed(parts) if p and not p.isdigit()), "")


def is_missing_like_value(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float):
        try:
            if pd.isna(value):
                return True
        except Exception:
            pass
    if isinstance(value, str):
        return value.strip().lower() in MISSING_LIKE_STRINGS
    try:
        if not isinstance(value, (list, dict, tuple, set)) and pd.isna(value):
            return True
    except Exception:
        pass
    if isinstance(value, (list, dict)) and len(value) == 0:
        return True
    return False


def is_empty_value(value: Any) -> bool:  # noqa: F811 - intentional notebook override
    return is_missing_like_value(value)


def requirement_allows_generic_context(req: Dict[str, Any], path: str) -> bool:
    leaf = path_leaf(path)
    if leaf not in GENERIC_CONTEXT_LEAVES:
        return True
    text = requirement_text_blob(req)
    return any(term in text for term in GENERIC_ALLOWED_TERMS)


def manual_route_bonus(req: Dict[str, Any], path: str) -> Tuple[int, List[str], bool]:
    rid = str(req.get("requirement_id", ""))
    if rid not in REQUIREMENT_ID_ROUTE_HINTS:
        return 0, [], False
    roots, hints = REQUIREMENT_ID_ROUTE_HINTS[rid]
    root = root_of_path(path)
    path_text = str(path).lower()
    matched = sorted([h for h in hints if h.lower() in path_text])
    if root in roots and matched:
        return 7, matched[:5], True
    if root in roots:
        return 3, [], True
    return -2, [], False


def evidence_candidate_allowed(req: Dict[str, Any], section_name: str, path: str, value: Any) -> Tuple[bool, str]:
    path = str(path)
    if any(fragment in path for fragment in NOISE_PATH_FRAGMENTS):
        return False, "excluded_noise_path"
    if is_missing_like_value(value):
        return False, "excluded_missing_like_value"
    if not requirement_allows_generic_context(req, path):
        return False, "excluded_generic_context_field"
    if root_of_path(path) == "metadata" and not metadata_allowed_for_requirement(req, path):
        return False, "excluded_metadata_not_relevant"
    return True, "allowed"


def evidence_score(req: Dict[str, Any], section_name: str, path: str, value: Any) -> Tuple[int, List[str], str]:  # noqa: F811
    allowed, reason = evidence_candidate_allowed(req, section_name, path, value)
    if not allowed:
        return -999, [], reason

    root = root_of_path(path)
    allowed_roots = allowed_roots_for_requirement(section_name, req)
    manual_bonus, manual_hits, manual_routed = manual_route_bonus(req, path)
    if manual_routed:
        allowed_roots = set(allowed_roots) | {root}

    req_kws = set(requirement_keywords(req))
    f_kws = set(field_keywords(path, value))
    overlap = sorted(req_kws.intersection(f_kws))

    score = len(overlap)
    path_lower = str(path).lower()
    for kw in req_kws:
        if len(kw) > 3 and kw in path_lower:
            score += 1

    route_reason = "lexical"
    if allowed_roots:
        if root in allowed_roots:
            score += 3
            route_reason = "payload_root_routing+lexical"
        else:
            score -= 4
            route_reason = "outside_expected_payload_root"

    boost, phrase_hits = phrase_path_boost(req, path, value)
    if boost:
        score += boost
        route_reason = "payload_root_routing+phrase_boost+lexical"

    if manual_bonus:
        score += manual_bonus
        route_reason = "manual_requirement_route+" + route_reason

    # V8.1 PATCH: do not boost generic reporting-year values.
    # Reporting-period fields can be contextual evidence but should not create
    # false "covered" decisions for broader disclosure clauses.

    matched = sorted(set(overlap + phrase_hits + manual_hits))
    return score, matched, route_reason


def evidence_strength(req: Dict[str, Any], candidate: Dict[str, Any]) -> str:
    score = int(candidate.get("match_score", 0) or 0)
    matched = candidate.get("matched_keywords", []) or []
    reason = str(candidate.get("mapping_reason", ""))

    # V8.1 PATCH:
    # - Generic context fields and audit-only paths can support context/scoring,
    #   but they are not enough to mark a requirement as fully covered.
    # - This prevents paths like reporting_year, summary_id, boundary_type or
    #   metadata.data_gaps[*] from upgrading coverage to "covered".
    if candidate.get("audit_only_evidence") or candidate.get("generic_context_field"):
        return "medium" if score >= 6 else "weak"

    if score >= 10 and (len(matched) >= 2 or "manual_requirement_route" in reason or "phrase_boost" in reason):
        return "strong"
    if score >= 6:
        return "medium"
    return "weak"


def make_candidate(req: Dict[str, Any], section_name: str, path: str, value: Any) -> Optional[Dict[str, Any]]:
    score, matched_terms, reason = evidence_score(req, section_name, path, value)
    if score < 6:
        return None
    candidate = {
        "payload_path": path,
        "payload_root": root_of_path(path),
        "value_preview": value_preview(value),
        "value_type": type(value).__name__,
        "match_score": score,
        "matched_keywords": matched_terms,
        "mapping_reason": reason,
        "generic_context_field": path_leaf(path) in GENERIC_CONTEXT_LEAVES,
        "audit_only_evidence": is_audit_only_evidence_path(path),
        "writer_safe": writer_evidence_path_allowed(path),
        "missing_like_value": False,
    }
    candidate["evidence_strength"] = evidence_strength(req, candidate)
    return candidate


def targeted_candidate_paths(req: Dict[str, Any], flat_payload: Dict[str, Any]) -> List[str]:
    rid = str(req.get("requirement_id", ""))
    if rid not in REQUIREMENT_ID_ROUTE_HINTS:
        return []
    roots, hints = REQUIREMENT_ID_ROUTE_HINTS[rid]
    out = []
    for path, value in flat_payload.items():
        if root_of_path(path) not in roots:
            continue
        if is_missing_like_value(value):
            continue
        path_lower = path.lower()
        if any(h.lower() in path_lower for h in hints):
            out.append(path)
    return out[:80]


def clean_and_enrich_evidence_map(section_name: str, evidence_map: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    reqs_by_id = {r["requirement_id"]: r for r in requirements_by_section[section_name]}
    payload = payloads_by_section[section_name]
    flat = flatten_json(payload)
    cleaned_map = []

    for row in evidence_map:
        req = reqs_by_id[row["requirement_id"]]
        by_path = {}

        # Re-score old candidates under strict rules.
        for c in row.get("evidence_candidates", []):
            path = c.get("payload_path")
            value = get_by_path(payload, path)
            candidate = make_candidate(req, section_name, path, value) if path else None
            if candidate:
                by_path[path] = candidate

        # Targeted enrichment for high-level clauses that lexical matching often misses.
        for path in targeted_candidate_paths(req, flat):
            if path in by_path:
                continue
            candidate = make_candidate(req, section_name, path, flat[path])
            if candidate:
                by_path[path] = candidate

        candidates = sorted(
            by_path.values(),
            key=lambda x: (
                2 if x.get("evidence_strength") == "strong" else 1 if x.get("evidence_strength") == "medium" else 0,
                x["match_score"],
                len(x.get("matched_keywords", [])),
                not x.get("generic_context_field", False),
            ),
            reverse=True,
        )[:8]

        cleaned_row = dict(row)
        cleaned_row["evidence_candidates"] = candidates
        cleaned_row["mapping_method"] = "deterministic_payload_aware_strict_v8_1_writer_safe_postprocessed"
        cleaned_map.append(cleaned_row)

    return cleaned_map


def classify_requirement_coverage(req: Dict[str, Any], mapping: Dict[str, Any]) -> Dict[str, Any]:  # noqa: F811
    candidates = mapping.get("evidence_candidates", [])
    mandatory = req.get("mandatory", True)
    strong_candidates = [c for c in candidates if c.get("evidence_strength") == "strong"]
    medium_candidates = [c for c in candidates if c.get("evidence_strength") == "medium"]
    max_score = max([c.get("match_score", 0) for c in candidates] or [0])

    if strong_candidates:
        status = "covered"
        selected = strong_candidates[:8]
    elif medium_candidates:
        status = "partially_covered"
        selected = medium_candidates[:8]
    else:
        selected = []
        if mandatory:
            status = "not_available_in_payload"
        else:
            raw_text = json.dumps(req.get("raw", {}), ensure_ascii=False).lower()
            status = "not_applicable" if any(x in raw_text for x in ["if applicable", "when applicable", "where applicable", "conditional"]) else "not_available_in_payload"

    return {
        "requirement_id": req["requirement_id"],
        "standard": req.get("standard", ""),
        "paragraph_id": req.get("paragraph_id", ""),
        "report_section": req.get("report_section", ""),
        "requirement_text": req.get("requirement_text", ""),
        "mandatory": mandatory,
        "coverage_status": status,
        "evidence_count": len(selected),
        "raw_candidate_count": len(candidates),
        "strong_candidate_count": len(strong_candidates),
        "medium_candidate_count": len(medium_candidates),
        "evidence_confidence_score": max_score,
        "evidence_paths": [c["payload_path"] for c in selected],
        "coverage_quality_note": "Strict V8.1 coverage: covered requires at least one strong, non-null, non-generic, writer-safe evidence candidate.",
        "not_applicable_justification": "Conditional/non-mandatory requirement with no relevant synthetic payload evidence." if status == "not_applicable" else "",
    }


def build_coverage_and_missing_register(section_name: str) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:  # noqa: F811
    reqs = requirements_by_section[section_name]
    maps = {m["requirement_id"]: m for m in evidence_maps_by_section[section_name]}
    coverage = []
    missing = []

    for req in reqs:
        cov = classify_requirement_coverage(req, maps[req["requirement_id"]])
        coverage.append(cov)
        if cov["coverage_status"] == "not_available_in_payload":
            missing.append({
                "flag_type": "missing_requirement",
                "requirement_id": cov["requirement_id"],
                "standard": cov["standard"],
                "paragraph_id": cov["paragraph_id"],
                "report_section": section_name,
                "mandatory": cov["mandatory"],
                "requirement_text": cov["requirement_text"],
                "coverage_status": "not_available_in_payload",
                "reason": "No sufficiently strong, non-null, non-generic payload evidence was identified for this requirement under strict V8.1 coverage rules.",
                "action_needed": "Add real evidence to the section payload, improve deterministic routing, or manually map an existing evidence path after review.",
                "report_instruction": "Do not mention this missing requirement or missing data in the generated report. Keep it only in audit outputs.",
            })

    counts = Counter([c["coverage_status"] for c in coverage])
    section_readiness_score = round(
        100 * (counts.get("covered", 0) + 0.5 * counts.get("partially_covered", 0)) / max(1, len(coverage)),
        2,
    )
    missing_register = {
        "section_name": section_name,
        "pipeline_mode": PIPELINE_MODE,
        "policy": "Missing requirements are recorded here and are excluded from report prose.",
        "missing_requirements_count": len(missing),
        "missing_requirement_ids": [m["requirement_id"] for m in missing],
        "section_readiness_score_0_to_100": section_readiness_score,
        "missing_requirements": missing,
    }
    return coverage, missing_register


def rebuild_strict_evidence_coverage_outputs() -> None:
    global evidence_maps_by_section, evidence_map_summaries, coverage_by_section, missing_registers_by_section

    evidence_map_summaries = {}
    coverage_by_section = {}
    missing_registers_by_section = {}

    for section in SECTIONS:
        evidence_maps_by_section[section] = clean_and_enrich_evidence_map(section, evidence_maps_by_section[section])
        summary = summarize_evidence_map(section, evidence_maps_by_section[section])
        evidence_map_summaries[section] = summary
        slug = SECTION_SLUGS[section]
        write_json(evidence_maps_by_section[section], DIRS["evidence_maps"] / f"evidence_map_{slug}.json")
        write_json(summary, DIRS["evidence_maps"] / f"evidence_map_summary_{slug}.json")

        coverage, missing_register = build_coverage_and_missing_register(section)
        coverage_by_section[section] = coverage
        missing_registers_by_section[section] = missing_register
        write_json(coverage, DIRS["coverage"] / f"coverage_matrix_{slug}.json")
        write_json(missing_register, DIRS["missing_requirements"] / f"missing_requirements_{slug}.json")
        counts = Counter([c["coverage_status"] for c in coverage])
        print(
            section,
            "strict candidates:", summary["requirements_with_candidates"], "/", summary["requirements_total"],
            "| coverage:", dict(counts),
            "| missing:", len(missing_register["missing_requirements"]),
            "| readiness:", missing_register["section_readiness_score_0_to_100"],
        )

    combined_missing = {
        "pipeline_mode": PIPELINE_MODE,
        "policy": "The report contains only evidence-supported disclosures. Missing requirements are stored here, used for scoring/flagging, and never included in report prose.",
        "total_missing_requirements": sum(len(m.get("missing_requirements", [])) for m in missing_registers_by_section.values()),
        "sections": missing_registers_by_section,
    }
    write_json(combined_missing, DIRS["missing_requirements"] / "missing_requirements_all_sections.json")


rebuild_strict_evidence_coverage_outputs()


General Requirements strict candidates: 98 / 108 | coverage: {'partially_covered': 45, 'not_available_in_payload': 10, 'covered': 53} | missing: 10 | readiness: 69.91
Governance strict candidates: 15 / 15 | coverage: {'covered': 14, 'partially_covered': 1} | missing: 0 | readiness: 96.67
Strategy strict candidates: 70 / 70 | coverage: {'covered': 54, 'partially_covered': 16} | missing: 0 | readiness: 88.57
Risk Management strict candidates: 17 / 17 | coverage: {'covered': 10, 'partially_covered': 7} | missing: 0 | readiness: 79.41
Metrics and Targets strict candidates: 135 / 151 | coverage: {'partially_covered': 10, 'covered': 125, 'not_available_in_payload': 16} | missing: 16 | readiness: 86.09


## Deterministic section planner

The planner is code-first. It builds a disclosure plan from covered requirements, available evidence, section blueprints, and table patterns. Missing requirements are excluded from the plan and kept only in JSON audit files.

In [11]:
# ============================================================
# CELL 10 — DETERMINISTIC SECTION PLANNER
# ============================================================

DEFAULT_SECTION_SUBSECTIONS = {
    "General Requirements": [
        {"heading": "Basis of preparation", "keywords": ["basis", "preparation", "compliance", "standard"]},
        {"heading": "Reporting boundary and connected information", "keywords": ["boundary", "entity", "connected", "financial"]},
        {"heading": "Materiality and judgement", "keywords": ["material", "judgement", "estimate", "assumption"]},
    ],
    "Governance": [
        {"heading": "Governance oversight", "keywords": ["board", "committee", "oversight", "governance"]},
        {"heading": "Roles, responsibilities and escalation", "keywords": ["responsibility", "role", "management", "escalation", "report"]},
        {"heading": "Skills, controls and monitoring", "keywords": ["skill", "competence", "control", "monitor", "training"]},
    ],
    "Strategy": [
        {"heading": "Business model and value chain", "keywords": ["business", "model", "value", "chain", "upstream", "downstream"]},
        {"heading": "Sustainability-related risks and opportunities", "keywords": ["risk", "opportunity", "material", "impact"]},
        {"heading": "Time horizons and financial effects", "keywords": ["time", "horizon", "financial", "cash", "performance"]},
        {"heading": "Resilience and strategic response", "keywords": ["resilience", "strategy", "response", "scenario"]},
    ],
    "Risk Management": [
        {"heading": "Risk identification and assessment", "keywords": ["identify", "assessment", "assess", "risk"]},
        {"heading": "Risk management processes and controls", "keywords": ["manage", "process", "control", "mitigation"]},
        {"heading": "Monitoring, reporting and integration", "keywords": ["monitor", "report", "integrat", "escalation"]},
    ],
    "Metrics and Targets": [
        {"heading": "Metrics register", "keywords": ["metric", "value", "unit", "measure"]},
        {"heading": "Targets and progress", "keywords": ["target", "baseline", "progress", "goal"]},
        {"heading": "Methodology and source traceability", "keywords": ["method", "source", "boundary", "definition"]},
    ],
}


def choose_subsection(section_name: str, requirement_text: str) -> str:
    req_tokens = set(tokens(requirement_text))
    candidates = DEFAULT_SECTION_SUBSECTIONS[section_name]
    scored = []
    for sub in candidates:
        score = sum(1 for kw in sub["keywords"] if any(kw in t for t in req_tokens))
        scored.append((score, sub["heading"]))
    scored.sort(reverse=True)
    return scored[0][1] if scored and scored[0][0] > 0 else candidates[0]["heading"]



# V8.3/V8.4 PATCH: aggressively sanitize authoring blueprints before they enter disclosure plans
# or writer context. Blueprint templates can contain generic layout guidance such
# as "missing data protocol" or "not currently available"; those are useful for
# generic reporting templates but must not reach this report because missing
# requirements are audit/scoring-only.
BLUEPRINT_PROSE_LEAKAGE_PATTERNS = [
    # Explicit missing-data / audit leakage
    "missing requirement",
    "missing data",
    "missing/not applicable",
    "data gap",
    "data gaps",
    "unavailable",
    "not available",
    "not currently available",
    "not reported",
    "not yet covered",
    "not applicable",
    "not material",
    "not currently reported",
    "no data",
    "no available data",
    "not enough data",
    "insufficient data",
    "insufficient evidence",
    "report_instruction",
    "not_available_in_payload",
    "metadata.data_gaps",
    "payload fields",

    # Generic blueprint phrasing that tends to make the writer add gap/limitation prose
    # even when the evidence pack is otherwise clean. These concepts stay audit-only
    # unless explicitly supported by a writer-safe evidence path and a normal requirement.
    "material gaps",
    "scope gaps",
    "gap or area",
    "gaps or areas",
    "current limitations",
    "limitations disclosure",
    "limitations and enhancement",
    "limitations and planned",
    "limitations and next",
    "scope/limitations",
    "key limitations",
    "limitations (",
    "limitation",
    "improvement plans",
    "planned enhancements",
    "future enhancements",
    "continuous improvement",
    "enhancement roadmap",
    "do not leave blanks",
    "status labels",
    "standardized status labels",
    "standardised status labels",
    "data/method constraints",
    "method constraints",
    "data constraints",
]

def blueprint_text_is_writer_safe(text: str) -> bool:
    lower = str(text).lower()
    return not any(pattern in lower for pattern in BLUEPRINT_PROSE_LEAKAGE_PATTERNS)

def sanitize_blueprint_for_report(obj: Any) -> Any:
    """Recursively remove blueprint instructions that could make the writer
    mention missing data, missing requirements, unavailable data, or audit-only
    information in report prose."""
    if isinstance(obj, dict):
        cleaned = {}
        for key, value in obj.items():
            # Remove entire key-value pairs when the key itself is unsafe.
            if not blueprint_text_is_writer_safe(key):
                continue
            cleaned_value = sanitize_blueprint_for_report(value)
            # Drop empty strings/lists/dicts created by filtering.
            if cleaned_value in ({}, [], ""):
                continue
            cleaned[key] = cleaned_value
        return cleaned
    if isinstance(obj, list):
        cleaned = []
        for item in obj:
            # If an item is a prose string and unsafe, remove it.
            if isinstance(item, str):
                if blueprint_text_is_writer_safe(item):
                    cleaned.append(item)
                continue
            # If a dict/list item serializes to unsafe prose, sanitize inside rather than
            # dropping the whole object unless it becomes empty.
            cleaned_item = sanitize_blueprint_for_report(item)
            if cleaned_item not in ({}, [], ""):
                # Defensive second check for fully textual small objects.
                serialized = json.dumps(cleaned_item, ensure_ascii=False)
                if blueprint_text_is_writer_safe(serialized):
                    cleaned.append(cleaned_item)
                else:
                    # Keep only if recursive cleaning removed explicit unsafe pieces;
                    # otherwise drop the object to avoid leakage.
                    if isinstance(cleaned_item, (dict, list)):
                        # If it still contains unsafe text after cleaning, skip.
                        continue
                    cleaned.append(cleaned_item)
        return cleaned
    if isinstance(obj, str):
        return obj if blueprint_text_is_writer_safe(obj) else ""
    return obj

def load_writer_safe_section_blueprint(section_name: str) -> Dict[str, Any]:
    return sanitize_blueprint_for_report(load_section_blueprint(section_name) or {})


def build_disclosure_plan(section_name: str) -> Dict[str, Any]:
    coverage = coverage_by_section[section_name]
    reqs_by_id = {r["requirement_id"]: r for r in requirements_by_section[section_name]}
    maps_by_id = {m["requirement_id"]: m for m in evidence_maps_by_section[section_name]}

    include_statuses = {"covered"}
    if ALLOW_PARTIAL_COVERAGE:
        include_statuses.add("partially_covered")

    supported = [c for c in coverage if c["coverage_status"] in include_statuses and c["evidence_count"] > 0]

    subsections = []
    subsection_map = defaultdict(lambda: {
        "heading": "",
        "purpose": "",
        "requirement_ids": [],
        "evidence_paths": [],
        "recommended_format": "short narrative",
    })

    for cov in supported:
        req = reqs_by_id[cov["requirement_id"]]
        heading = choose_subsection(section_name, req["requirement_text"])
        item = subsection_map[heading]
        item["heading"] = heading
        item["purpose"] = f"Address evidence-supported {section_name.lower()} disclosure requirements related to {heading.lower()}."
        item["requirement_ids"].append(cov["requirement_id"])
        # V8.1 PATCH: do not pass audit-only evidence, such as metadata.data_gaps,
        # to disclosure plans or the section writer. Missing-data details stay in
        # audit/scoring JSON only.
        item["evidence_paths"].extend([p for p in cov["evidence_paths"] if writer_evidence_path_allowed(p)])

    for heading, item in subsection_map.items():
        item["requirement_ids"] = sorted(set(item["requirement_ids"]))
        item["evidence_paths"] = sorted(set(item["evidence_paths"]))
        if section_name == "Metrics and Targets":
            item["recommended_format"] = "table-first with brief narrative"
        elif section_name in {"Governance", "Risk Management"}:
            item["recommended_format"] = "narrative plus responsibility/process table if evidence supports it"
        elif section_name == "Strategy":
            item["recommended_format"] = "structured narrative plus value-chain/time-horizon table if evidence supports it"
        subsections.append(dict(item))

    if not subsections:
        subsections = [{
            "heading": section_name,
            "purpose": "No evidence-supported requirements were available for report drafting.",
            "requirement_ids": [],
            "evidence_paths": [],
            "recommended_format": "omit section content or mark for human review",
        }]

    # Recommended tables from style table patterns.
    recommended_tables = []
    recommended_columns = TABLE_PATTERNS.get("recommended_columns_by_table_type", {}) if isinstance(TABLE_PATTERNS, dict) else {}
    for table_name, cols in recommended_columns.items():
        t = table_name.lower()
        if section_name.lower().split()[0] in t or (
            section_name == "Metrics and Targets" and "metrics" in t
        ) or (
            section_name == "Risk Management" and "risk" in t
        ):
            recommended_tables.append({"table_name": table_name, "columns": cols})

    plan = {
        "section_name": section_name,
        "pipeline_mode": PIPELINE_MODE,
        "policy": "Plan includes only covered/partially covered evidence-supported requirements. Audit-only evidence paths are excluded from report content.",
        "subsections": subsections,
        # V8.4 PATCH: sanitize style-derived recommended_tables too.
        # These templates can contain optional columns such as "Notes on scope/limitations",
        # which can make the writer add limitation/missing-data prose even when the
        # evidence pack is clean.
        "recommended_tables": sanitize_blueprint_for_report(recommended_tables)[:4],
        "section_blueprint": load_writer_safe_section_blueprint(section_name),
    }
    return plan

plans_by_section = {}
for section in SECTIONS:
    plan = build_disclosure_plan(section)
    plans_by_section[section] = plan
    write_json(plan, DIRS["plans"] / f"disclosure_plan_{SECTION_SLUGS[section]}.json")
    print(section, "subsections:", len(plan["subsections"]), "recommended tables:", len(plan["recommended_tables"]))

General Requirements subsections: 3 recommended tables: 1
Governance subsections: 2 recommended tables: 2
Strategy subsections: 4 recommended tables: 1
Risk Management subsections: 2 recommended tables: 3
Metrics and Targets subsections: 3 recommended tables: 1


## LLM writer and claims builder

The writer only receives supported requirements and supported evidence. It must not mention missing requirements, synthetic data, missing payloads, or unavailable information.

In [12]:
# ============================================================
# CELL 11 — CONTEXT PACKER FOR LLM AGENTS
# ============================================================


def requirement_subset(section_name: str, requirement_ids: List[str]) -> List[Dict[str, Any]]:
    reqs = {r["requirement_id"]: r for r in requirements_by_section[section_name]}
    return [reqs[rid] for rid in requirement_ids if rid in reqs]


def evidence_subset(section_name: str, evidence_paths: List[str], limit_value_chars: int = 500) -> List[Dict[str, Any]]:
    payload = payloads_by_section[section_name]
    out = []
    for path in sorted(set(evidence_paths)):
        # V8.1 PATCH: writer and claims agents must never receive audit-only
        # paths such as metadata.data_gaps[*]. Missing information is handled
        # only in audit/scoring outputs.
        if not writer_evidence_path_allowed(path):
            continue
        value = get_by_path(payload, path)
        if value is not None and not is_missing_like_value(value):
            out.append({
                "payload_path": path,
                "value_preview": value_preview(value, limit=limit_value_chars),
                "value_type": type(value).__name__,
            })
    return out


def build_writer_context(section_name: str) -> Dict[str, Any]:
    plan = plans_by_section[section_name]
    req_ids = sorted(set([rid for sub in plan["subsections"] for rid in sub.get("requirement_ids", [])]))
    ev_paths = sorted(set([p for sub in plan["subsections"] for p in sub.get("evidence_paths", [])]))
    return {
        "section_name": section_name,
        "pipeline_mode": PIPELINE_MODE,
        "critical_rules": [
            "Write only evidence-supported disclosures.",
            "Use only supported_requirements and evidence_items; ignore audit files and audit-only evidence paths.",
            "Do not discuss absent, unsupported, omitted, unprovided, or audit-only items in report prose.",
            "Do not invent committees, policies, tools, targets, metrics, dates, currencies, financial effects, or maturity claims.",
            "Do not use the PDF layout guide or emit layout placeholders.",
            "Use the target payload only as factual evidence; style guides affect wording only.",
        ],
        "supported_requirements": requirement_subset(section_name, req_ids),
        "disclosure_plan": plan,
        "evidence_items": evidence_subset(section_name, ev_paths),
        "authoring_style": GLOBAL_STYLE,
        "section_style": load_section_style(section_name),
        "section_blueprint": load_writer_safe_section_blueprint(section_name),
        "table_patterns": TABLE_PATTERNS,
        "no_copying_rules": NO_COPYING_RULES,
    }


def truncate_context(obj: Any, max_chars: int = 60000) -> str:
    text = json.dumps(obj, ensure_ascii=False, indent=2)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n...TRUNCATED_FOR_TOKEN_LIMIT..."

In [13]:
# ============================================================
# CELL 12 — SECTION WRITER AGENT
# ============================================================


def write_section_draft(section_name: str) -> Dict[str, Any]:
    context = build_writer_context(section_name)

    system = """
You are an IFRS S1/S2 sustainability disclosure writer.
You write audit-ready Markdown sections using only the provided evidence.
You must not invent facts. You must not mention missing requirements, missing data, unavailable data, missing payload fields, data gaps, or synthetic-data limitations in report prose.
Return JSON only.
""".strip()

    user = f"""
Write the {section_name} section in Markdown.

Rules:
1. Use only supported_requirements and evidence_items.
2. Do not disclose or mention requirements that are missing from the payload/audit register.
3. Do not write phrases such as "not available", "unavailable", "missing", "data gap", "not provided", "synthetic dataset", "payload", or "missing requirement".
4. Do not use PDF layout placeholders such as divider pages, image placeholders, or page spreads.
5. Use neutral, IFRS-aligned, non-promotional language.
6. Use tables only when evidence supports table content.
7. Target-company-specific names are allowed only if present in evidence_items.
8. Return valid JSON with keys: section_name, draft_markdown, writer_notes.

Context:
{truncate_context(context)}
""".strip()

    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["section_writer"],
        temperature=0.15,
        max_tokens=6000,
        response_format={"type": "json_object"},
    )
    obj = parse_json_response(raw, request_label=f"section_writer_{SECTION_SLUGS[section_name]}")
    obj.setdefault("section_name", section_name)
    obj.setdefault("draft_markdown", "")
    return obj

In [14]:

# ============================================================
# CELL 12B — V8.5 WRITER CONTEXT + PREFLIGHT PATCH
# ============================================================
# Why this patch exists:
# - V8.4 disclosure plans were clean, but the section writer still produced
#   generic template tables, [Insert ...] placeholders, and "missing data" prose.
# - Strategy also produced a "no source content" paragraph because the original
#   writer context placed long requirements/plans before the actual evidence.
#
# Fix:
# - Put evidence_items first in the writer context.
# - Remove generic table/style blueprints from writer context.
# - Force real evidence-derived rows only; no placeholders.
# - Run a local writer preflight and retry once before returning the draft.
# ============================================================

DRAFT_PLACEHOLDER_REGEX = re.compile(r"\[[^\]]+\]")

WRITER_UNSAFE_PHRASES = [
    "[insert",
    "insert risk/opportunity",
    "insert metric",
    "insert definition",
    "insert value",
    "placeholder",
    "missing or incomplete data",
    "missing data",
    "incomplete data",
    "not reported for the period",
    "not reported",
    "not available",
    "unavailable",
    "data not available",
    "methodology under development",
    "boundary not yet defined",
    "planned improvement direction",
    "source content",
    "provided source content",
    "intentionally limited",
    "no entity-specific",
    "no entity specific",
    "missing requirement",
    "data gap",
    "data gaps",
    "payload",
    "synthetic",
]

def compact_supported_requirements_for_writer(section_name: str, requirement_ids: List[str]) -> List[Dict[str, Any]]:
    """Return compact requirements only; no raw requirement metadata."""
    req_by_id = {r["requirement_id"]: r for r in requirements_by_section[section_name]}
    cov_by_id = {c["requirement_id"]: c for c in coverage_by_section.get(section_name, [])}
    out = []
    for rid in requirement_ids:
        req = req_by_id.get(rid)
        if not req:
            continue
        cov = cov_by_id.get(rid, {})
        out.append({
            "requirement_id": rid,
            "standard": req.get("standard", ""),
            "paragraph_id": req.get("paragraph_id", ""),
            "coverage_status": cov.get("coverage_status", ""),
            "requirement_text": req.get("requirement_text", "")[:900],
        })
    return out

def compact_plan_for_writer(plan: Dict[str, Any]) -> Dict[str, Any]:
    """Keep only authoring structure; avoid generic blueprint/table templates."""
    return {
        "section_name": plan.get("section_name", ""),
        "policy": plan.get("policy", ""),
        "subsections": [
            {
                "heading": sub.get("heading", ""),
                "purpose": sub.get("purpose", ""),
                "requirement_ids": sub.get("requirement_ids", []),
                "recommended_format": sub.get("recommended_format", ""),
            }
            for sub in plan.get("subsections", [])
        ],
        "recommended_tables": plan.get("recommended_tables", [])[:2],
    }

def evidence_summary_by_root(evidence_items: List[Dict[str, Any]], max_per_root: int = 40) -> Dict[str, List[Dict[str, Any]]]:
    grouped = defaultdict(list)
    for item in evidence_items:
        path = item.get("payload_path", "")
        root = path.split("[", 1)[0].split(".", 1)[0] if path else "unknown"
        if len(grouped[root]) < max_per_root:
            grouped[root].append(item)
    return dict(grouped)

def build_writer_context(section_name: str) -> Dict[str, Any]:  # noqa: F811 - intentional V8.5 override
    plan = plans_by_section[section_name]
    req_ids = sorted(set([rid for sub in plan["subsections"] for rid in sub.get("requirement_ids", [])]))
    ev_paths = sorted(set([p for sub in plan["subsections"] for p in sub.get("evidence_paths", [])]))
    evidence_items = evidence_subset(section_name, ev_paths, limit_value_chars=700)

    return {
        "section_name": section_name,
        "hard_writer_rules": [
            "Write only report-ready prose using evidence_items.",
            "Use the actual values in evidence_items. Do not output template rows.",
            "Never write bracketed placeholders such as [Insert ...].",
            "Never write phrases about missing data, unavailable data, not reported items, payloads, source-content absence, or synthetic data.",
            "If evidence is partial, write only the supported subset; do not explain what is absent.",
            "If a table cannot be populated with actual evidence-derived rows, omit the table and use concise narrative.",
            "Do not state that no evidence exists when evidence_items is non-empty.",
            "Do not invent committees, policies, tools, targets, metrics, dates, currencies, financial effects, or maturity claims.",
            "Use neutral IFRS-aligned language and keep claims traceable to evidence_items.",
        ],
        # Put actual evidence before requirements/plans to avoid truncation hiding the facts.
        "evidence_items": evidence_items,
        "evidence_summary_by_root": evidence_summary_by_root(evidence_items),
        "supported_requirements": compact_supported_requirements_for_writer(section_name, req_ids),
        "disclosure_plan": compact_plan_for_writer(plan),
        "style_guidance": {
            "tone": "audit-ready, neutral, concise, non-promotional",
            "tables": "Use tables only with real evidence values; no placeholders.",
            "citations": "Do not include paragraph citations in report prose unless explicitly requested elsewhere.",
        },
    }

def writer_preflight_issues(section_name: str, draft_markdown: str, evidence_items: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
    text = draft_markdown or ""
    lower = text.lower()
    issues = []

    bracket_hits = DRAFT_PLACEHOLDER_REGEX.findall(text)
    if bracket_hits:
        issues.append({
            "type": "placeholder_brackets",
            "count": len(bracket_hits),
            "examples": bracket_hits[:10],
        })

    phrase_hits = [p for p in WRITER_UNSAFE_PHRASES if p in lower]
    if phrase_hits:
        issues.append({
            "type": "unsafe_report_language",
            "phrases": phrase_hits,
        })

    if evidence_items is None:
        evidence_items = []
    word_count = len(re.findall(r"\b\w+\b", text))
    if evidence_items and word_count < 120:
        issues.append({
            "type": "too_short_given_available_evidence",
            "word_count": word_count,
            "evidence_item_count": len(evidence_items),
        })

    # Generic empty tables are usually template leakage.
    if "| [insert" in lower or lower.count("[insert") >= 2:
        issues.append({
            "type": "generic_template_table",
            "message": "Draft contains unpopulated template rows.",
        })

    return issues

def write_section_draft(section_name: str) -> Dict[str, Any]:  # noqa: F811 - intentional V8.5 override
    context = build_writer_context(section_name)

    system = """
You are an IFRS S1/S2 sustainability disclosure writer.
You write final report-ready Markdown using only evidence_items.
You must not invent facts.
You must not mention missing requirements, missing data, unavailable data, not-reported items, absent source content, payloads, synthetic data, or data gaps.
You must not output placeholders or unpopulated template tables.
Return JSON only.
""".strip()

    base_user = f"""
Write the {section_name} section in Markdown.

Rules:
1. Use only evidence_items and supported_requirements.
2. The section must contain actual evidence-derived content, not generic templates.
3. Do not output [Insert ...], placeholder rows, empty tables, or instructions to the reporting entity.
4. Do not write phrases such as "not available", "unavailable", "missing", "not reported", "data gap", "payload", "synthetic", "source content", or "no entity-specific".
5. If a table is used, every row must be populated from evidence_items. Otherwise omit the table.
6. Do not say that evidence is absent when evidence_items is non-empty.
7. Use neutral, concise, IFRS-aligned language.
8. Return valid JSON with keys: section_name, draft_markdown, writer_notes.

Context:
{truncate_context(context, max_chars=100000)}
""".strip()

    last_obj = None
    last_issues = []

    for attempt in range(2):
        if attempt == 0:
            user = base_user
        else:
            user = f"""
The previous draft failed local preflight and must be rewritten.

Preflight issues:
{json.dumps(last_issues, ensure_ascii=False, indent=2)}

Rewrite the {section_name} section. Follow these additional rules:
- Remove all placeholder/template text.
- Remove all missing-data/not-available/source-content language.
- Use actual evidence values from evidence_items.
- If evidence is partial, write only supported facts without discussing absent facts.
- If a populated table is not possible, write concise narrative instead.

Context:
{truncate_context(context, max_chars=100000)}
""".strip()

        raw = azure_chat(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            model_tier=MODEL_CONFIG["section_writer"],
            temperature=0.05,
            max_tokens=6500,
            response_format={"type": "json_object"},
        )
        obj = parse_json_response(raw, request_label=f"section_writer_{SECTION_SLUGS[section_name]}_attempt{attempt}")
        obj.setdefault("section_name", section_name)
        obj.setdefault("draft_markdown", "")
        last_obj = obj
        last_issues = writer_preflight_issues(section_name, obj.get("draft_markdown", ""), context.get("evidence_items", []))
        obj["writer_preflight_issues"] = last_issues
        if not last_issues:
            return obj

    return last_obj or {"section_name": section_name, "draft_markdown": "", "writer_preflight_issues": last_issues}


In [15]:

# ============================================================
# CELL 12C — V8.6 EXPANDED REPORT WRITER PATCH
# ============================================================
# Why this patch exists:
# - V8.5 fixed hallucination, placeholders, and missing-data language.
# - The resulting drafts were safe, but several sections felt truncated and
#   summary-like instead of final-report-like.
#
# Fix:
# - Preserve all V8.5 safety rules.
# - Add controlled expansion requirements: richer narrative, explicit evidence
#   explanation, pillar connectivity, and report-quality subsection depth.
# - Add section-specific minimum word targets to prevent approval of overly thin
#   drafts when evidence exists.
# ============================================================

SECTION_EXPANSION_TARGETS = {
    "General Requirements": {
        "min_words": 700,
        "target_words": "800-1,100",
        "depth_focus": [
            "basis of preparation, reporting period, currency and comparatives",
            "material sustainability-related information and why it matters to prospects",
            "connected information across governance, strategy, risk management, and metrics",
            "measurement approaches, assumptions, judgement and data-quality characteristics",
            "comparative consistency and change monitoring",
        ],
    },
    "Governance": {
        "min_words": 800,
        "target_words": "900-1,300",
        "depth_focus": [
            "board oversight, agenda integration and reporting flow",
            "management-level responsibility and committee structures",
            "ERM and major-transaction climate checks",
            "skills, competence and development programme",
            "executive remuneration linkage and how governance information supports decision-making",
        ],
    },
    "Strategy": {
        "min_words": 1_100,
        "target_words": "1,200-1,800",
        "depth_focus": [
            "identified risks and opportunities with time horizons",
            "effects on business model and value chain",
            "strategic response and decision-making trade-offs",
            "scenario resilience findings and transmission channels",
            "financial planning/resource allocation evidence and progress monitoring",
        ],
    },
    "Risk Management": {
        "min_words": 850,
        "target_words": "900-1,300",
        "depth_focus": [
            "risk identification and assessment lifecycle",
            "inputs, data sources, scenario links and rating methodology",
            "prioritisation relative to other risks and ERM integration",
            "monitoring frequencies and changed-since-prior-period indicators",
            "value-chain risk considerations and opportunity handling",
        ],
    },
    "Metrics and Targets": {
        "min_words": 900,
        "target_words": "1,000-1,500",
        "depth_focus": [
            "reporting boundary and period",
            "financed emissions metrics and methodology",
            "operational GHG emissions and Scope 2 treatment",
            "targets, milestones, progress, validation and carbon credits",
            "internal carbon price and financed-emissions data-quality mix",
        ],
    },
}

# Keep the V8.5 unsafe phrases and add a few report-depth specific blockers.
WRITER_UNSAFE_PHRASES = sorted(set(WRITER_UNSAFE_PHRASES + [
    "section is intentionally limited",
    "intentionally limited to evidence-supported",
    "no source evidence",
    "no source content",
    "no provided evidence",
    "not enough evidence",
    "insufficient evidence",
    "cannot be determined",
    "could not be determined",
    "template",
]))

# These generic internal-field names can appear in evidence paths, but final prose
# should translate them into readable report language.
RAW_FIELDNAME_PROSE_PATTERNS = [
    r"\bclimate_risk_register\.",
    r"\berm_integrated_flag\b",
    r"\bchanged_since_prior_period\b",
    r"\bscope2_market_tco2e\b",
    r"\bscope2_location_tco2e\b",
]


def section_word_count(markdown: str) -> int:
    return len(re.findall(r"\b\w+\b", markdown or ""))


def section_expansion_profile(section_name: str) -> Dict[str, Any]:
    return SECTION_EXPANSION_TARGETS.get(section_name, {
        "min_words": 700,
        "target_words": "800-1,200",
        "depth_focus": ["explain evidence in report-ready narrative", "connect the section to other disclosure pillars"],
    })


def build_writer_context(section_name: str) -> Dict[str, Any]:  # noqa: F811 - intentional V8.6 override
    plan = plans_by_section[section_name]
    req_ids = sorted(set([rid for sub in plan["subsections"] for rid in sub.get("requirement_ids", [])]))
    ev_paths = sorted(set([p for sub in plan["subsections"] for p in sub.get("evidence_paths", [])]))
    evidence_items = evidence_subset(section_name, ev_paths, limit_value_chars=900)
    expansion = section_expansion_profile(section_name)

    return {
        "section_name": section_name,
        "hard_writer_rules": [
            "Write only final-report prose using evidence_items.",
            "Use actual values in evidence_items; no template rows, placeholder text, or instructions to the reporting entity.",
            "Never write bracketed placeholders such as [Insert ...].",
            "Never write phrases about missing data, unavailable data, not reported items, payloads, source-content absence, synthetic data, or data gaps.",
            "If evidence is partial, write only the supported subset; do not explain what is absent.",
            "Do not state that no evidence exists when evidence_items is non-empty.",
            "Do not invent committees, policies, tools, targets, metrics, dates, currencies, financial effects, or maturity claims.",
            "Use neutral IFRS-aligned language and keep claims traceable to evidence_items.",
            "Expansion means explaining and connecting supported facts; it never means adding unsupported facts.",
        ],
        "expansion_requirements": {
            "minimum_word_count": expansion["min_words"],
            "target_word_range": expansion["target_words"],
            "depth_focus": expansion["depth_focus"],
            "subsection_pattern": [
                "Start each major subsection with a purpose or framing sentence.",
                "Then explain the evidence in report language, using exact supported values where relevant.",
                "Add one connectivity sentence when evidence supports links to governance, strategy, risk management, or metrics/targets.",
                "Use tables only if they can be populated entirely from evidence_items; otherwise use narrative or bullets.",
                "Avoid raw internal field names in prose; translate them into readable disclosure language.",
            ],
        },
        # Put actual evidence before requirements/plans to avoid truncation hiding facts.
        "evidence_items": evidence_items,
        "evidence_summary_by_root": evidence_summary_by_root(evidence_items, max_per_root=80),
        "supported_requirements": compact_supported_requirements_for_writer(section_name, req_ids),
        "disclosure_plan": compact_plan_for_writer(plan),
        "style_guidance": {
            "tone": "audit-ready, neutral, connected, report-like, non-promotional",
            "depth": "Do not produce a short evidence summary. Produce a complete section narrative using the target word range.",
            "tables": "Use tables only with real evidence values; no placeholders.",
            "citations": "Do not include paragraph citations in report prose unless explicitly requested elsewhere.",
        },
    }


def writer_preflight_issues(section_name: str, draft_markdown: str, evidence_items: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:  # noqa: F811
    text = draft_markdown or ""
    lower = text.lower()
    issues = []

    bracket_hits = DRAFT_PLACEHOLDER_REGEX.findall(text)
    if bracket_hits:
        issues.append({
            "type": "placeholder_brackets",
            "count": len(bracket_hits),
            "examples": bracket_hits[:10],
        })

    phrase_hits = [p for p in WRITER_UNSAFE_PHRASES if p in lower]
    if phrase_hits:
        issues.append({
            "type": "unsafe_report_language",
            "phrases": phrase_hits,
        })

    if evidence_items is None:
        evidence_items = []
    word_count = section_word_count(text)
    min_words = int(section_expansion_profile(section_name).get("min_words", 700))
    if evidence_items and word_count < min_words:
        issues.append({
            "type": "too_short_truncated_section",
            "word_count": word_count,
            "minimum_word_count": min_words,
            "evidence_item_count": len(evidence_items),
            "instruction": "Expand using existing evidence only; do not add unsupported facts or missing-data language.",
        })

    if "| [insert" in lower or lower.count("[insert") >= 2:
        issues.append({
            "type": "generic_template_table",
            "message": "Draft contains unpopulated template rows.",
        })

    raw_field_hits = []
    for pattern in RAW_FIELDNAME_PROSE_PATTERNS:
        if re.search(pattern, text):
            raw_field_hits.append(pattern)
    if raw_field_hits:
        issues.append({
            "type": "raw_field_names_in_report_prose",
            "patterns": raw_field_hits,
            "instruction": "Translate internal field names into readable report language.",
        })

    return issues


def write_section_draft(section_name: str) -> Dict[str, Any]:  # noqa: F811 - intentional V8.6 override
    context = build_writer_context(section_name)
    expansion = context["expansion_requirements"]

    system = """
You are a senior IFRS S1/S2 sustainability disclosure writer.
You write final, report-ready Markdown using only evidence_items.
You must not invent facts.
You must not mention missing requirements, missing data, unavailable data, not-reported items, absent source content, payloads, synthetic data, or data gaps.
You must not output placeholders or unpopulated template tables.
You must produce a complete section, not a short evidence summary.
Return JSON only.
""".strip()

    base_user = f"""
Write the {section_name} section in Markdown.

Depth target:
- Target length: {expansion['target_word_range']} words.
- Minimum acceptable length: {expansion['minimum_word_count']} words.
- Focus the expansion on: {json.dumps(expansion['depth_focus'], ensure_ascii=False)}.

Rules:
1. Use only evidence_items and supported_requirements.
2. Produce final-report narrative, not a compact evidence summary.
3. For each major topic, write a short framing paragraph and then explain the supported evidence.
4. Add connectivity sentences between this section and other pillars only when the evidence supports the connection.
5. Use tables only when every row can be populated from evidence_items; otherwise use paragraphs and bullets.
6. Do not output [Insert ...], placeholders, empty tables, instructions, or generic templates.
7. Do not write phrases such as "not available", "unavailable", "missing", "not reported", "data gap", "payload", "synthetic", "source content", "no entity-specific", or "insufficient evidence".
8. If evidence is partial, write only what is supported and do not discuss absent facts.
9. Translate internal field names into readable report language.
10. Use neutral, IFRS-aligned, non-promotional language.
11. Return valid JSON with keys: section_name, draft_markdown, writer_notes.

Context:
{truncate_context(context, max_chars=120000)}
""".strip()

    last_obj = None
    last_issues = []

    for attempt in range(3):
        if attempt == 0:
            user = base_user
        else:
            user = f"""
The previous draft failed local preflight and must be rewritten or expanded.

Preflight issues:
{json.dumps(last_issues, ensure_ascii=False, indent=2)}

Rewrite the {section_name} section with these corrections:
- Expand the section to at least {expansion['minimum_word_count']} words using existing evidence only.
- Do not add unsupported facts.
- Remove all placeholder/template text.
- Remove all missing-data/not-available/source-content language.
- Translate internal field names into readable report language.
- Keep all claims traceable to evidence_items.

Context:
{truncate_context(context, max_chars=120000)}
""".strip()

        raw = azure_chat(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            model_tier=MODEL_CONFIG["section_writer"],
            temperature=0.08,
            max_tokens=9000,
            response_format={"type": "json_object"},
        )
        obj = parse_json_response(raw, request_label=f"section_writer_{SECTION_SLUGS[section_name]}_v86_attempt{attempt}")
        obj.setdefault("section_name", section_name)
        obj.setdefault("draft_markdown", "")
        last_obj = obj
        last_issues = writer_preflight_issues(section_name, obj.get("draft_markdown", ""), context.get("evidence_items", []))
        obj["writer_preflight_issues"] = last_issues
        obj["writer_depth_profile"] = {
            "word_count": section_word_count(obj.get("draft_markdown", "")),
            "minimum_word_count": expansion["minimum_word_count"],
            "target_word_range": expansion["target_word_range"],
        }
        if not last_issues:
            return obj

    return last_obj or {"section_name": section_name, "draft_markdown": "", "writer_preflight_issues": last_issues}


In [16]:
# ============================================================
# CELL 13 — CLAIMS REGISTER BUILDER AGENT
# V8 PATCH:
# - Uses azure_chat_json(), so malformed/truncated JSON is repaired automatically.
# - Adds compact-output instructions to reduce JSON truncation risk.
# - Adds a deterministic fallback register so the pipeline does not crash if the
#   claims-builder output is unrecoverable.
# ============================================================


def _split_markdown_into_claim_sentences(markdown: str, max_claims: int = 80) -> List[str]:
    """Lightweight fallback splitter for material factual claims."""
    text = re.sub(r"```.*?```", " ", str(markdown or ""), flags=re.DOTALL)
    text = re.sub(r"\|", " ", text)  # tables become readable text
    text = re.sub(r"[#*_>`\[\]()]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    parts = re.split(r"(?<=[.!?])\s+|\s+;\s+", text)
    material = []
    for part in parts:
        s = part.strip(" -•\t\n")
        if len(s) < 35:
            continue
        lower = s.lower()
        looks_material = (
            bool(extract_numbers(s))
            or any(k in lower for k in [
                "board", "committee", "risk", "scenario", "scope", "emission", "target",
                "metric", "climate", "governance", "transition", "physical", "assurance",
                "financial", "greenhouse", "ghg", "remuneration", "oversight",
            ])
        )
        if looks_material:
            material.append(s[:800])
        if len(material) >= max_claims:
            break
    return material


def build_fallback_claims_register(section_name: str, draft_markdown: str, reason: str = "") -> Dict[str, Any]:
    """
    Last-resort deterministic claims register.

    It intentionally leaves evidence_sources empty and supported=False. That is
    safer than pretending support exists: deterministic gates/reviser can then
    remove or repair unsupported prose instead of crashing the notebook.
    """
    claims = []
    for i, sentence in enumerate(_split_markdown_into_claim_sentences(draft_markdown), start=1):
        claims.append({
            "claim_id": f"FALLBACK_CLAIM_{i:03d}",
            "claim_text": sentence,
            "claim_type": "fallback_extracted_sentence",
            "entities": extract_entities(sentence),
            "numbers": extract_numbers(sentence),
            "dates": [],
            "evidence_sources": [],
            "requirement_ids": [],
            "supported": False,
            "support_notes": (
                "Fallback register created because LLM claims-register JSON could not be parsed. "
                "No evidence source was assigned automatically."
            ),
        })

    return {
        "section_name": section_name,
        "claims": claims,
        "claims_register_warning": "deterministic_fallback_used",
        "fallback_reason": str(reason)[:1200],
    }


def build_claims_register(section_name: str, draft_markdown: str) -> Dict[str, Any]:
    plan = plans_by_section[section_name]
    req_ids = sorted(set([rid for sub in plan["subsections"] for rid in sub.get("requirement_ids", [])]))
    ev_paths = sorted(set([p for sub in plan["subsections"] for p in sub.get("evidence_paths", [])]))

    context = {
        "section_name": section_name,
        "draft_markdown": draft_markdown,
        "supported_requirements": requirement_subset(section_name, req_ids),
        "evidence_items": evidence_subset(section_name, ev_paths, limit_value_chars=700),
        "instructions": [
            "Extract material factual claims from the draft. Do not include purely generic wording.",
            "Keep claim_text concise: one sentence or less, max 45 words.",
            "For each claim, list entities, numbers, dates, evidence_sources and requirement_ids.",
            "evidence_sources must be exact payload_path values from evidence_items.",
            "If a claim has no evidence source, mark supported=false and explain why briefly.",
            "Do not create evidence paths that are not in evidence_items.",
            "Return compact valid JSON only. No markdown fences. No trailing commas.",
        ],
    }

    system = "You are a strict audit claims-register builder. Return compact valid JSON only."
    user = f"""
Build a claims register for this generated report section.

Return JSON with keys:
- section_name
- claims: list of objects with claim_id, claim_text, claim_type, entities, numbers, dates, evidence_sources, requirement_ids, supported, support_notes

Strict JSON rules:
- Output one complete JSON object only.
- Use double quotes for all keys and strings.
- Escape quotes inside strings.
- Do not end arrays/objects with trailing commas.
- Keep the register compact enough to finish completely.

Context:
{truncate_context(context)}
""".strip()

    try:
        obj = azure_chat_json(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            model_tier=MODEL_CONFIG["claims_register_builder"],
            temperature=0,
            max_tokens=int(os.getenv("CLAIMS_REGISTER_MAX_TOKENS", "9000")),
            request_label=f"claims_register_builder_{SECTION_SLUGS[section_name]}",
        )
    except Exception as exc:
        print("Claims register builder failed after JSON repair. Using deterministic fallback register.")
        obj = build_fallback_claims_register(section_name, draft_markdown, reason=repr(exc))

    obj.setdefault("section_name", section_name)
    obj.setdefault("claims", [])

    # Defensive normalization: the model may return evidence_sources as dicts
    # like {"payload_path": "..."} instead of plain path strings.
    # normalize_claims_register is defined in the deterministic gates cell and
    # is available by the time this function is called in the full pipeline.
    if "normalize_claims_register" in globals():
        obj = normalize_claims_register(obj)

    return obj


## Deterministic gates

These gates run before LLM judges and after every revision:

1. Claims integrity gate.
2. Factlock number/entity gate.
3. Reference firewall gate.
4. Report cleanliness gate.

If deterministic gates fail, the pipeline revises or escalates without wasting judge calls.

In [17]:
# ============================================================
# CELL 14 — DETERMINISTIC GATES
# V7 PATCH:
# - Adds deterministic gate diagnostics.
# - Repairs claims-register evidence_sources when the LLM gives incomplete paths.
# - Treats claims-register formatting issues as warnings instead of blocking
#   the whole pipeline when payload factlock is still satisfied.
# ============================================================

NUMBER_PATTERN = re.compile(
    r"(?<![A-Za-z0-9])(?:\d{1,3}(?:[, ]\d{3})+|\d+)(?:\.\d+)?\s?(?:%|bps|AED|USD|EUR|tCO2e|tonnes|years?|days?)?",
    flags=re.IGNORECASE,
)

ENTITY_PATTERN = re.compile(
    r"\b(?:[A-Z][A-Za-z0-9&\-/]+(?:\s+[A-Z][A-Za-z0-9&\-/]+){1,6})\b"
)

REPORT_CLEANLINESS_BLOCKLIST = [
    "synthetic dataset",
    "synthetic data",
    "synthetic payload",
    "missing from the payload",
    "not available in the payload",
    "not included in the payload",
    "payload does not include",
    "data not provided",
    "missing requirement",
]

REPORT_CLEANLINESS_SOFT_PHRASES = [
    # These can be legitimate methodology limitation language when instructed
    # by the payload metadata, so they are recorded as warnings, not blockers.
    "not available for prior years",
    "data unavailable for prior years",
    "prior years not available",
    "vehicle activity data available for 2024 only",
    "travel records cover 2024 only",
]


def extract_numbers(text: str) -> List[str]:
    return sorted(set([m.group(0).strip() for m in NUMBER_PATTERN.finditer(text)]))


def extract_entities(text: str) -> List[str]:
    raw = [m.group(0).strip() for m in ENTITY_PATTERN.finditer(text)]
    ignore = {
        "IFRS", "IFRS S1", "IFRS S2", "General Requirements",
        "Risk Management", "Metrics and Targets", "Scope 1", "Scope 2", "Scope 3",
        "Table", "Figure"
    }
    return sorted(set([
        x for x in raw
        if x not in ignore
        and not x.startswith("Table ")
        and not x.startswith("Figure ")
    ]))


def payload_text(section_name: str) -> str:
    return json.dumps(payloads_by_section[section_name], ensure_ascii=False)


def _extract_path_from_evidence_source(src: Any) -> Optional[str]:
    """
    Normalize LLM evidence source shapes to a string payload path.
    Accepted examples:
    - "payload.path[0].field"
    - {"payload_path": "payload.path[0].field", ...}
    - {"path": "..."} / {"evidence_path": "..."} / {"source_path": "..."}
    """
    if src is None:
        return None

    if isinstance(src, str):
        path = src.strip()
        return path or None

    if isinstance(src, dict):
        for key in ("payload_path", "path", "evidence_path", "source_path", "payloadPath", "source"):
            value = src.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip()

        # Last-resort recursive search for a payload-like path string.
        for value in src.values():
            if isinstance(value, str):
                candidate = value.strip()
                if re.search(r"^[A-Za-z_][A-Za-z0-9_]*(?:\[\d+\])?(?:\.[A-Za-z_][A-Za-z0-9_]*(?:\[\d+\])?)*$", candidate):
                    return candidate

    return None


def _normalize_evidence_sources(value: Any) -> List[str]:
    """Return a clean list of payload path strings from arbitrary LLM output."""
    if value is None:
        return []

    if isinstance(value, (str, dict)):
        path = _extract_path_from_evidence_source(value)
        return [path] if path else []

    if isinstance(value, list):
        out = []
        for item in value:
            path = _extract_path_from_evidence_source(item)
            if path:
                out.append(path)
        return sorted(set(out))

    return []


def _normalize_string_list(value: Any, preferred_keys: Optional[List[str]] = None) -> List[str]:
    """
    Normalize LLM-produced list fields such as entities, numbers, dates,
    and requirement_ids. Handles scalar strings, lists, and dict items.
    """
    if preferred_keys is None:
        preferred_keys = ["value", "text", "name", "id", "requirement_id", "number", "date", "entity"]

    if value is None:
        return []

    if isinstance(value, (str, int, float, bool)):
        text = str(value).strip()
        return [text] if text else []

    if isinstance(value, dict):
        for key in preferred_keys:
            item = value.get(key)
            if item is not None:
                text = str(item).strip()
                return [text] if text else []
        return []

    if isinstance(value, list):
        out = []
        for item in value:
            out.extend(_normalize_string_list(item, preferred_keys=preferred_keys))
        return sorted(set([x for x in out if x]))

    return []


def normalize_claims_register(claims_register: Dict[str, Any]) -> Dict[str, Any]:
    """
    Makes the claims register deterministic-gate safe without changing its meaning.
    It prevents crashes when the LLM returns dicts instead of plain strings.
    """
    if not isinstance(claims_register, dict):
        return {"claims": []}

    claims = claims_register.get("claims", [])
    if isinstance(claims, dict):
        claims = list(claims.values())
    if not isinstance(claims, list):
        claims = []

    normalized_claims = []
    for i, claim in enumerate(claims, start=1):
        if not isinstance(claim, dict):
            continue

        c = dict(claim)
        c.setdefault("claim_id", f"CLAIM_{i:03d}")

        c["claim_text"] = str(c.get("claim_text", "")).strip()
        c["evidence_sources"] = _normalize_evidence_sources(c.get("evidence_sources", []))
        c["requirement_ids"] = _normalize_string_list(
            c.get("requirement_ids", []),
            preferred_keys=["requirement_id", "id", "value", "text"],
        )
        c["numbers"] = _normalize_string_list(
            c.get("numbers", []),
            preferred_keys=["number", "value", "text"],
        )
        c["entities"] = _normalize_string_list(
            c.get("entities", []),
            preferred_keys=["entity", "name", "value", "text"],
        )
        c["dates"] = _normalize_string_list(
            c.get("dates", []),
            preferred_keys=["date", "value", "text"],
        )

        normalized_claims.append(c)

    out = dict(claims_register)
    out["claims"] = normalized_claims
    return out


def _candidate_evidence_paths_for_section(section_name: str) -> List[str]:
    """Evidence paths allowed for the section from the deterministic disclosure plan."""
    plan = plans_by_section.get(section_name, {})
    paths = []
    for sub in plan.get("subsections", []):
        paths.extend(sub.get("evidence_paths", []))
    return sorted(set([p for p in paths if isinstance(p, str) and p.strip()]))


def _score_claim_against_payload_value(claim: Dict[str, Any], path: str, value: Any) -> float:
    """Score how likely a payload path supports a claim."""
    claim_text = str(claim.get("claim_text", ""))
    path_text = path.replace("_", " ").replace(".", " ")
    value_text = value_preview(value, limit=1200)

    claim_tokens = set(tokens(claim_text))
    evidence_tokens = set(tokens(path_text + " " + value_text))

    score = 0.0
    score += 1.5 * len(claim_tokens & evidence_tokens)

    # Exact numbers are highly valuable.
    for num in claim.get("numbers", []):
        n = str(num).strip()
        if n and n.lower() in value_text.lower():
            score += 10

    # Exact entities are valuable too.
    for ent in claim.get("entities", []):
        e = str(ent).strip().lower()
        if e and (e in value_text.lower() or e in path.lower()):
            score += 8

    # Some short claims contain no extracted numbers/entities but mention key concepts.
    lower_claim = claim_text.lower()
    lower_ev = (path_text + " " + value_text).lower()
    for phrase in [
        "reporting entity", "reporting year", "reporting period", "financial control",
        "assurance", "board", "committee", "remuneration", "scenario", "risk rating",
        "scope 1", "scope 2", "scope 3", "financed emissions", "carbon intensity",
        "target", "baseline", "greenhouse gas", "transition", "physical risk",
    ]:
        if phrase in lower_claim and phrase in lower_ev:
            score += 4

    return score


def repair_claim_evidence_sources(section_name: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    """
    Repair claims-register evidence paths using the deterministic disclosure plan.

    This does not invent facts. It only attaches existing allowed payload paths
    from the section plan when they clearly overlap with a claim.
    """
    payload = payloads_by_section[section_name]
    candidate_paths = _candidate_evidence_paths_for_section(section_name)

    if not candidate_paths:
        return normalize_claims_register(claims_register)

    claims_register = normalize_claims_register(claims_register)
    claims = claims_register.get("claims", [])

    for claim in claims:
        # Keep only sources that really resolve.
        valid_sources = []
        invalid_sources = []
        for src in claim.get("evidence_sources", []):
            if get_by_path(payload, src) is not None:
                valid_sources.append(src)
            else:
                invalid_sources.append(src)

        # Add repairs if there are no valid sources.
        repairs = []
        if not valid_sources:
            scored = []
            for path in candidate_paths:
                value = get_by_path(payload, path)
                if value is None or is_empty_value(value):
                    continue
                score = _score_claim_against_payload_value(claim, path, value)
                if score >= 8:
                    scored.append((score, path))
            scored.sort(reverse=True)
            repairs = [path for _, path in scored[:3]]

        claim["evidence_sources"] = sorted(set(valid_sources + repairs))
        if repairs:
            claim["evidence_repair_note"] = "Added by deterministic path repair from disclosure-plan evidence paths."
            claim["repaired_evidence_sources"] = repairs
        if invalid_sources:
            claim["invalid_evidence_sources_removed"] = invalid_sources

    out = dict(claims_register)
    out["claims"] = claims
    return out


def claims_integrity_gate(section_name: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    payload = payloads_by_section[section_name]
    req_ids = {r["requirement_id"] for r in requirements_by_section[section_name]}

    failures = []
    warnings = []

    claims_register = repair_claim_evidence_sources(section_name, claims_register)
    claims = claims_register.get("claims", [])

    for claim in claims:
        cid = claim.get("claim_id", "UNKNOWN")
        claim_text = claim.get("claim_text", "")

        if claim.get("supported") is False:
            failures.append({
                "claim_id": cid,
                "issue": "claim_marked_unsupported",
                "claim": claim_text,
            })

        evidence_sources = claim.get("evidence_sources", [])
        if not evidence_sources:
            # This is usually a claims-builder formatting failure. Keep as a warning
            # and allow factlock to decide whether unsupported numbers/entities exist.
            warnings.append({
                "claim_id": cid,
                "issue": "no_evidence_source_after_repair",
                "claim": claim_text,
            })

        for src in evidence_sources:
            if not isinstance(src, str) or not src.strip():
                warnings.append({
                    "claim_id": cid,
                    "issue": "invalid_evidence_source_shape",
                    "evidence_source": repr(src)[:500],
                })
                continue

            if get_by_path(payload, src) is None:
                failures.append({
                    "claim_id": cid,
                    "issue": "evidence_source_does_not_resolve",
                    "evidence_source": src,
                })

        valid_rids = []
        for rid in claim.get("requirement_ids", []):
            if rid in req_ids:
                valid_rids.append(rid)
            else:
                warnings.append({
                    "claim_id": cid,
                    "issue": "unknown_requirement_id_ignored",
                    "requirement_id": rid,
                })
        claim["requirement_ids"] = valid_rids

    return {
        "gate_name": "claims_integrity",
        "passed": len(failures) == 0,
        "failures": failures,
        "warnings": warnings[:100],
        "claim_count": len(claims),
        "claims_register_normalized": claims_register,
    }


def _compact_number(value: str) -> str:
    """
    Normalize number strings for payload matching:
    '1,154.8 tCO2e' -> '1154.8'
    '27.4%' -> '27.4'
    """
    v = str(value).lower()
    v = re.sub(r"(tco2e|tonnes|years?|days?|bps|eur|usd|aed|%)", "", v, flags=re.I)
    v = v.replace(",", "").replace(" ", "").strip()
    return v


def _payload_number_index(section_name: str) -> set:
    """Build normalized scalar number index from the payload."""
    payload = payloads_by_section[section_name]
    flat = flatten_json(payload)
    idx = set()
    for value in flat.values():
        if isinstance(value, (int, float)):
            idx.add(_compact_number(str(value)))
        elif isinstance(value, str):
            for num in extract_numbers(value):
                idx.add(_compact_number(num))
    return idx


def factlock_gate(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    claims_register = repair_claim_evidence_sources(section_name, claims_register)

    draft_numbers = extract_numbers(draft_markdown)
    draft_entities = extract_entities(draft_markdown)

    claim_numbers = set()
    claim_entities = set()
    for claim in claims_register.get("claims", []):
        claim_numbers.update([str(x).strip() for x in claim.get("numbers", [])])
        claim_entities.update([str(x).strip() for x in claim.get("entities", [])])

    p_text = payload_text(section_name).lower()
    payload_num_index = _payload_number_index(section_name)
    failures = []
    warnings = []

    for num in draft_numbers:
        raw = num.strip()
        n = raw.lower()
        compact = _compact_number(raw)

        # Pure table/section numbering can pass.
        if re.fullmatch(r"\d+(\.\d+)?", raw):
            continue

        if n in p_text or raw in claim_numbers or compact in payload_num_index:
            continue

        failures.append({"type": "number_not_in_payload_or_claims", "value": raw})

    allowed_entities = {
        "ifrs s1", "ifrs s2", "ifrs sustainability disclosure standards",
        "general requirements", "governance", "strategy",
        "risk management", "metrics and targets", "scope 1", "scope 2", "scope 3"
    }

    for ent in draft_entities:
        ent_l = ent.lower().strip()
        if ent_l in allowed_entities:
            continue
        if ent_l not in p_text and ent not in claim_entities:
            # Some title-case phrases are headings, not factual entities.
            if any(ent_l == term for term in allowed_entities):
                continue
            warnings.append({"type": "entity_not_in_payload_or_claims", "value": ent})

    # Entity warnings are not blockers because headings and IFRS phraseology cause
    # many false positives. Numbers remain blocking.
    return {
        "gate_name": "factlock_numbers_entities",
        "passed": len(failures) == 0,
        "failures": failures[:100],
        "warnings": warnings[:100],
        "draft_numbers": draft_numbers,
        "draft_entities": draft_entities[:100],
    }


def reference_firewall_gate(draft_markdown: str) -> Dict[str, Any]:
    text_l = draft_markdown.lower()
    hits = [term for term in FORBIDDEN_TERMS if term and term.lower() in text_l]
    return {
        "gate_name": "reference_firewall",
        "passed": len(hits) == 0,
        "forbidden_term_hits": hits,
        "failures": [{"type": "forbidden_reference_term", "value": h} for h in hits],
        "warnings": [],
    }


def report_cleanliness_gate(draft_markdown: str) -> Dict[str, Any]:
    text_l = draft_markdown.lower()
    hard_hits = [phrase for phrase in REPORT_CLEANLINESS_BLOCKLIST if phrase in text_l]
    soft_hits = [phrase for phrase in REPORT_CLEANLINESS_SOFT_PHRASES if phrase in text_l]
    return {
        "gate_name": "report_cleanliness_no_missing_payload_language",
        "passed": len(hard_hits) == 0,
        "blocked_phrase_hits": hard_hits,
        "soft_phrase_hits": soft_hits,
        "failures": [{"type": "blocked_report_phrase", "value": h} for h in hard_hits],
        "warnings": [{"type": "soft_report_phrase_check_manually", "value": h} for h in soft_hits],
    }


def summarize_deterministic_failures(deterministic: Dict[str, Any], max_items: int = 5) -> str:
    """Create a compact console-friendly deterministic gate summary."""
    lines = []
    for gate in deterministic.get("gates", []):
        failures = gate.get("failures", []) or []
        warnings = gate.get("warnings", []) or []
        status = "PASS" if gate.get("passed") else "FAIL"
        lines.append(f"- {gate.get('gate_name')}: {status} | failures={len(failures)} | warnings={len(warnings)}")
        for f in failures[:max_items]:
            lines.append(f"  failure: {str(f)[:500]}")
        for w in warnings[:min(2, max_items)]:
            lines.append(f"  warning: {str(w)[:500]}")
    return "\n".join(lines)


def run_deterministic_gates(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    # Normalize and repair once here, then pass the same cleaned object to gates.
    cleaned_claims = repair_claim_evidence_sources(section_name, claims_register)

    gates = [
        claims_integrity_gate(section_name, cleaned_claims),
        factlock_gate(section_name, draft_markdown, cleaned_claims),
        reference_firewall_gate(draft_markdown),
        report_cleanliness_gate(draft_markdown),
    ]

    result = {
        "section_name": section_name,
        "passed": all(g["passed"] for g in gates),
        "gates": gates,
        "summary": None,
    }
    result["summary"] = summarize_deterministic_failures(result)
    return result


# V8 strict report cleanliness: the final report must not contain any missing-data language.
REPORT_CLEANLINESS_BLOCKLIST = sorted(set(REPORT_CLEANLINESS_BLOCKLIST + REPORT_CLEANLINESS_SOFT_PHRASES + [
    "missing data",
    "data gap",
    "data gaps",
    "unavailable data",
    "data unavailable",
    "not available",
    "not provided",
    "not disclosed due to missing",
    "no data",
    "payload",
    "synthetic",
    "human review required",
]))
REPORT_CLEANLINESS_SOFT_PHRASES = []


In [18]:

# ============================================================
# CELL 14B — V8.5 STRUCTURAL QUALITY GATE PATCH
# ============================================================
# Adds hard deterministic failures for:
# - [Insert ...] / template placeholders
# - missing-data / not-reported / source-content prose
# - ultra-short "no source content" sections when evidence exists
# ============================================================

# Extend global cleanliness blocklist before pipeline scoring uses it.
REPORT_CLEANLINESS_BLOCKLIST = sorted(set(REPORT_CLEANLINESS_BLOCKLIST + [
    "[insert",
    "insert risk/opportunity",
    "insert metric",
    "insert definition",
    "insert value",
    "placeholder",
    "missing or incomplete data",
    "incomplete data",
    "not reported for the period",
    "not reported",
    "methodology under development",
    "boundary not yet defined",
    "planned improvement direction",
    "source content",
    "provided source content",
    "intentionally limited",
    "no entity-specific",
    "no entity specific",
]))

def draft_structural_quality_gate(section_name: str, draft_markdown: str) -> Dict[str, Any]:
    failures = []
    warnings = []
    text = draft_markdown or ""
    lower = text.lower()

    bracket_hits = DRAFT_PLACEHOLDER_REGEX.findall(text) if "DRAFT_PLACEHOLDER_REGEX" in globals() else re.findall(r"\[[^\]]+\]", text)
    if bracket_hits:
        failures.append({
            "type": "placeholder_brackets",
            "count": len(bracket_hits),
            "examples": bracket_hits[:10],
        })

    phrase_hits = [p for p in WRITER_UNSAFE_PHRASES if p in lower] if "WRITER_UNSAFE_PHRASES" in globals() else []
    if phrase_hits:
        failures.append({
            "type": "unsafe_report_language",
            "phrases": phrase_hits,
        })

    evidence_path_count = len(_candidate_evidence_paths_for_section(section_name))
    word_count = len(re.findall(r"\b\w+\b", text))
    if evidence_path_count > 0 and word_count < 120:
        failures.append({
            "type": "too_short_given_available_evidence",
            "word_count": word_count,
            "evidence_path_count": evidence_path_count,
        })

    return {
        "gate_name": "draft_structural_quality_no_templates_no_absence_language",
        "passed": len(failures) == 0,
        "failures": failures,
        "warnings": warnings,
    }

def run_deterministic_gates(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:  # noqa: F811
    cleaned_claims = repair_claim_evidence_sources(section_name, claims_register)

    gates = [
        draft_structural_quality_gate(section_name, draft_markdown),
        claims_integrity_gate(section_name, cleaned_claims),
        factlock_gate(section_name, draft_markdown, cleaned_claims),
        reference_firewall_gate(draft_markdown),
        report_cleanliness_gate(draft_markdown),
    ]

    result = {
        "section_name": section_name,
        "passed": all(g["passed"] for g in gates),
        "gates": gates,
        "summary": None,
    }
    result["summary"] = summarize_deterministic_failures(result)
    return result


In [19]:

# ============================================================
# CELL 14C — V8.6 DEPTH / NON-TRUNCATION GATE PATCH
# ============================================================
# Adds deterministic failures for safe-but-truncated drafts.
# This gate is intentionally placed after V8.5 so it overrides the structural
# quality gate and deterministic-gate runner.
# ============================================================

REPORT_CLEANLINESS_BLOCKLIST = sorted(set(REPORT_CLEANLINESS_BLOCKLIST + [
    "section is intentionally limited",
    "intentionally limited to evidence-supported",
    "no source evidence",
    "no source content",
    "no provided evidence",
    "not enough evidence",
    "insufficient evidence",
    "cannot be determined",
    "could not be determined",
]))


def draft_depth_quality_gate(section_name: str, draft_markdown: str) -> Dict[str, Any]:
    failures = []
    warnings = []
    text = draft_markdown or ""
    lower = text.lower()
    word_count = section_word_count(text) if "section_word_count" in globals() else len(re.findall(r"\b\w+\b", text))
    evidence_path_count = len(_candidate_evidence_paths_for_section(section_name))
    profile = section_expansion_profile(section_name) if "section_expansion_profile" in globals() else {"min_words": 700}
    min_words = int(profile.get("min_words", 700))

    if evidence_path_count >= 10 and word_count < min_words:
        failures.append({
            "type": "section_too_short_or_truncated",
            "word_count": word_count,
            "minimum_word_count": min_words,
            "evidence_path_count": evidence_path_count,
            "required_fix": "Expand the section using existing evidence only; do not add unsupported facts or missing-data language.",
        })

    # Detect raw internal field names in prose. Evidence paths are acceptable in
    # audit files, but final report prose should be human-readable.
    raw_hits = []
    for pattern in RAW_FIELDNAME_PROSE_PATTERNS if "RAW_FIELDNAME_PROSE_PATTERNS" in globals() else []:
        if re.search(pattern, text):
            raw_hits.append(pattern)
    if raw_hits:
        failures.append({
            "type": "raw_field_names_in_report_prose",
            "patterns": raw_hits,
            "required_fix": "Translate internal payload field names into readable report language.",
        })

    # Too many ultra-short subsections is another truncation signal.
    headings = re.split(r"\n###\s+", text)
    short_blocks = []
    for block in headings[1:]:
        title = block.splitlines()[0].strip() if block.splitlines() else ""
        wc = len(re.findall(r"\b\w+\b", block))
        if wc and wc < 55:
            short_blocks.append({"heading": title, "word_count": wc})
    if len(short_blocks) >= 3 and evidence_path_count >= 20:
        warnings.append({
            "type": "many_short_subsections",
            "examples": short_blocks[:5],
            "suggested_fix": "Add explanatory narrative to each supported subsection.",
        })

    return {
        "gate_name": "draft_depth_quality_no_truncation",
        "passed": len(failures) == 0,
        "failures": failures,
        "warnings": warnings,
    }


def draft_structural_quality_gate(section_name: str, draft_markdown: str) -> Dict[str, Any]:  # noqa: F811
    failures = []
    warnings = []
    text = draft_markdown or ""
    lower = text.lower()

    bracket_hits = DRAFT_PLACEHOLDER_REGEX.findall(text) if "DRAFT_PLACEHOLDER_REGEX" in globals() else re.findall(r"\[[^\]]+\]", text)
    if bracket_hits:
        failures.append({
            "type": "placeholder_brackets",
            "count": len(bracket_hits),
            "examples": bracket_hits[:10],
        })

    phrase_hits = [p for p in WRITER_UNSAFE_PHRASES if p in lower] if "WRITER_UNSAFE_PHRASES" in globals() else []
    if phrase_hits:
        failures.append({
            "type": "unsafe_report_language",
            "phrases": phrase_hits,
        })

    # Preserve V8.5 minimum, but V8.6 depth gate handles stronger thresholds.
    evidence_path_count = len(_candidate_evidence_paths_for_section(section_name))
    word_count = len(re.findall(r"\b\w+\b", text))
    if evidence_path_count > 0 and word_count < 120:
        failures.append({
            "type": "too_short_given_available_evidence",
            "word_count": word_count,
            "evidence_path_count": evidence_path_count,
        })

    return {
        "gate_name": "draft_structural_quality_no_templates_no_absence_language",
        "passed": len(failures) == 0,
        "failures": failures,
        "warnings": warnings,
    }


def run_deterministic_gates(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:  # noqa: F811
    cleaned_claims = repair_claim_evidence_sources(section_name, claims_register)

    gates = [
        draft_structural_quality_gate(section_name, draft_markdown),
        draft_depth_quality_gate(section_name, draft_markdown),
        claims_integrity_gate(section_name, cleaned_claims),
        factlock_gate(section_name, draft_markdown, cleaned_claims),
        reference_firewall_gate(draft_markdown),
        report_cleanliness_gate(draft_markdown),
    ]

    result = {
        "section_name": section_name,
        "passed": all(g["passed"] for g in gates),
        "gates": gates,
        "summary": None,
    }
    result["summary"] = summarize_deterministic_failures(result)
    return result


In [20]:
# ============================================================
# CELL 15 — LLM JUDGES
# ============================================================


def judge_ifrs_coverage(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    context = {
        "section_name": section_name,
        "draft_markdown": draft_markdown,
        "coverage_matrix": coverage_by_section[section_name],
        "missing_requirements_policy": "Missing requirements must be absent from report prose and present in missing_requirements.json.",
        "missing_requirements_register": missing_registers_by_section[section_name],
        "claims_register": claims_register,
    }
    system = "You are an IFRS S1/S2 coverage judge. Return JSON only."
    user = f"""
Judge the generated section against available IFRS requirements.

Important policy:
- Do NOT fail the section because requirements marked not_available_in_payload are absent from the report.
- Fail if a missing requirement or any missing-data/unavailable-data wording is invented or mentioned in the report.
- Fail if a covered requirement is not addressed despite available evidence.
- Missing requirements must be tracked in missing_requirements_register, not in report prose.

Return JSON with: approved, ifrs_coverage_score_0_to_10, missing_supported_requirements, invented_missing_requirements, missing_data_language_found, required_fixes, summary.

Context:
{truncate_context(context)}
""".strip()
    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["ifrs_coverage_judge"],
        temperature=0,
        max_tokens=3500,
        response_format={"type": "json_object"},
    )
    return parse_json_response(raw, request_label=f"ifrs_coverage_judge_{SECTION_SLUGS[section_name]}")


def judge_evidence_support(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    plan = plans_by_section[section_name]
    ev_paths = sorted(set([p for sub in plan["subsections"] for p in sub.get("evidence_paths", [])]))
    context = {
        "section_name": section_name,
        "draft_markdown": draft_markdown,
        "claims_register": claims_register,
        "evidence_items": evidence_subset(section_name, ev_paths, limit_value_chars=900),
        "rules": [
            "Every material claim must be supported by payload evidence.",
            "No invented metrics, targets, committees, policies, tools, dates, currencies, or financial effects.",
            "Do not penalize omission of missing requirements listed in missing_requirements.json.",
        ],
    }
    system = "You are a strict evidence support judge. Return JSON only."
    user = f"""
Judge whether the section contains unsupported claims.

Return JSON with: approved, evidence_score_0_to_10, unsupported_claims, questionable_claims, required_fixes, summary.

Context:
{truncate_context(context)}
""".strip()
    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["evidence_judge"],
        temperature=0,
        max_tokens=3500,
        response_format={"type": "json_object"},
    )
    return parse_json_response(raw, request_label=f"evidence_judge_{SECTION_SLUGS[section_name]}")


def judge_style(section_name: str, draft_markdown: str) -> Dict[str, Any]:
    context = {
        "section_name": section_name,
        "draft_markdown": draft_markdown,
        "global_style_guide": GLOBAL_STYLE,
        "section_style": load_section_style(section_name),
        "style_compliance_rubric": STYLE_RUBRIC,
        "no_copying_rules": NO_COPYING_RULES,
    }
    system = "You are a sustainability report style judge. Return JSON only."
    user = f"""
Judge whether the section follows the approved authoring style.

Return JSON with: approved, style_score_0_to_10, voice_issues, structure_issues, wording_issues, table_figure_issues, required_fixes, summary.

Context:
{truncate_context(context)}
""".strip()
    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["style_judge"],
        temperature=0,
        max_tokens=3000,
        response_format={"type": "json_object"},
    )
    return parse_json_response(raw, request_label=f"style_judge_{SECTION_SLUGS[section_name]}")


def run_llm_judges(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "section_name": section_name,
        "ifrs_coverage_judge": judge_ifrs_coverage(section_name, draft_markdown, claims_register),
        "evidence_judge": judge_evidence_support(section_name, draft_markdown, claims_register),
        "style_judge": judge_style(section_name, draft_markdown),
    }

In [21]:
# ============================================================
# CELL 16 — COMPOSITE APPROVAL GATE
# ============================================================

APPROVAL_THRESHOLDS = {
    "ifrs_coverage_score_min": float(os.getenv("IFRS_COVERAGE_SCORE_MIN", "8.0")),
    "evidence_score_min": float(os.getenv("EVIDENCE_SCORE_MIN", "8.0")),
    "style_score_min": float(os.getenv("STYLE_SCORE_MIN", "7.5")),
}


def _score(obj: Dict[str, Any], *names: str) -> float:
    for name in names:
        if name in obj:
            try:
                return float(obj[name])
            except Exception:
                pass
    return 0.0


def composite_approval_gate(
    section_name: str,
    deterministic_result: Dict[str, Any],
    judge_results: Optional[Dict[str, Any]],
) -> Dict[str, Any]:
    if not deterministic_result.get("passed", False):
        return {
            "section_name": section_name,
            "approved": False,
            "reason": "deterministic_gates_failed",
            "required_fixes": deterministic_result,
        }

    if judge_results is None:
        return {
            "section_name": section_name,
            "approved": False,
            "reason": "llm_judges_not_run",
            "required_fixes": [],
        }

    ifrs = judge_results.get("ifrs_coverage_judge", {})
    evidence = judge_results.get("evidence_judge", {})
    style = judge_results.get("style_judge", {})

    ifrs_score = _score(ifrs, "ifrs_coverage_score_0_to_10", "score")
    evidence_score = _score(evidence, "evidence_score_0_to_10", "score")
    style_score = _score(style, "style_score_0_to_10", "score")

    failures = []
    if ifrs_score < APPROVAL_THRESHOLDS["ifrs_coverage_score_min"] or not ifrs.get("approved", False):
        failures.append({"judge": "ifrs_coverage_judge", "score": ifrs_score, "required_fixes": ifrs.get("required_fixes", [])})
    if evidence_score < APPROVAL_THRESHOLDS["evidence_score_min"] or not evidence.get("approved", False):
        failures.append({"judge": "evidence_judge", "score": evidence_score, "required_fixes": evidence.get("required_fixes", [])})
    if style_score < APPROVAL_THRESHOLDS["style_score_min"] or not style.get("approved", False):
        failures.append({"judge": "style_judge", "score": style_score, "required_fixes": style.get("required_fixes", [])})

    return {
        "section_name": section_name,
        "approved": len(failures) == 0,
        "scores": {
            "ifrs_coverage": ifrs_score,
            "evidence": evidence_score,
            "style": style_score,
        },
        "failures": failures,
        "thresholds": APPROVAL_THRESHOLDS,
    }

In [22]:
# ============================================================
# CELL 17 — MINIMAL REVISER AGENT
# ============================================================


def collect_fix_instructions(deterministic_result: Dict[str, Any], judge_results: Optional[Dict[str, Any]], approval: Dict[str, Any]) -> Dict[str, Any]:
    fixes = {
        "deterministic_gate_failures": [],
        "judge_required_fixes": [],
        "approval_failures": approval.get("failures", []),
    }
    if not deterministic_result.get("passed", False):
        fixes["deterministic_gate_failures"] = deterministic_result.get("gates", [])

    if judge_results:
        for judge_name, result in judge_results.items():
            if isinstance(result, dict):
                fixes["judge_required_fixes"].append({
                    "judge": judge_name,
                    "approved": result.get("approved"),
                    "required_fixes": result.get("required_fixes", []),
                    "summary": result.get("summary", ""),
                })
    return fixes


def revise_section_minimally(
    section_name: str,
    draft_markdown: str,
    claims_register: Dict[str, Any],
    deterministic_result: Dict[str, Any],
    judge_results: Optional[Dict[str, Any]],
    approval: Dict[str, Any],
) -> Dict[str, Any]:
    context = build_writer_context(section_name)
    fix_instructions = collect_fix_instructions(deterministic_result, judge_results, approval)

    reviser_context = {
        "section_name": section_name,
        "current_draft_markdown": draft_markdown,
        "current_claims_register": claims_register,
        "fix_instructions": fix_instructions,
        "allowed_context": context,
        "hard_rules": [
            "Revise minimally.",
            "Do not add new facts or claims.",
            "Do not mention missing requirements, missing payload, unavailable data, or synthetic data in the report.",
            "Remove unsupported claims rather than inventing support.",
            "Use only evidence_items already provided.",
            "Return JSON only with keys: section_name, revised_markdown, revision_notes.",
        ],
    }

    system = "You are a minimal IFRS disclosure reviser. Return JSON only."
    user = f"""
Revise the section to fix the listed issues.

Context:
{truncate_context(reviser_context)}
""".strip()

    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["minimal_reviser"],
        temperature=0.05,
        max_tokens=6000,
        response_format={"type": "json_object"},
    )
    obj = parse_json_response(raw, request_label=f"minimal_reviser_{SECTION_SLUGS[section_name]}")
    obj.setdefault("section_name", section_name)
    obj.setdefault("revised_markdown", draft_markdown)
    return obj

## Section pipeline loop

The loop writes one section, builds its claims register, runs deterministic gates, runs LLM judges only when the deterministic gates pass, and revises up to `MAX_REVISION_LOOPS`.

In [23]:
# ============================================================
# CELL 18 — RUN ONE SECTION PIPELINE
# V8 PATCH: saves section-generation scores and blocks missing-data prose.
# ============================================================

MISSING_DATA_REPORT_TERMS = sorted(set(REPORT_CLEANLINESS_BLOCKLIST + [
    "missing", "unavailable", "not available", "not provided", "data gap", "data gaps", "payload", "synthetic"
]))


def scan_for_missing_data_language(markdown: str) -> List[Dict[str, Any]]:
    text_lower = str(markdown).lower()
    hits = []
    for term in MISSING_DATA_REPORT_TERMS:
        term_l = term.lower()
        if term_l in text_lower:
            hits.append({"term": term, "issue": "missing_data_language_in_report_prose"})
    return hits


def coverage_score_for_section(section_name: str) -> Dict[str, Any]:
    coverage = coverage_by_section.get(section_name, [])
    counts = Counter([c.get("coverage_status") for c in coverage])
    total = len(coverage)
    weighted = counts.get("covered", 0) + 0.5 * counts.get("partially_covered", 0)
    score = round(100 * weighted / max(1, total), 2)
    return {
        "requirements_total": total,
        "coverage_counts": dict(counts),
        "coverage_score_0_to_100": score,
    }


def score_section_generation_output(
    section_name: str,
    draft_markdown: str,
    deterministic: Dict[str, Any],
    judges: Optional[Dict[str, Any]],
    approval: Dict[str, Any],
) -> Dict[str, Any]:
    coverage_component = coverage_score_for_section(section_name)
    missing_register = missing_registers_by_section.get(section_name, {})
    missing_count = missing_register.get("missing_requirements_count", len(missing_register.get("missing_requirements", [])))
    missing_hits = scan_for_missing_data_language(draft_markdown)

    deterministic_score = 100.0 if deterministic.get("passed", False) else 0.0
    cleanliness_score = 0.0 if missing_hits else 100.0

    if judges:
        ifrs_score = _score(judges.get("ifrs_coverage_judge", {}), "ifrs_coverage_score_0_to_10", "score") * 10
        evidence_score = _score(judges.get("evidence_judge", {}), "evidence_score_0_to_10", "score") * 10
        style_score = _score(judges.get("style_judge", {}), "style_score_0_to_10", "score") * 10
    else:
        ifrs_score = evidence_score = style_score = 0.0

    judge_average = round((ifrs_score + evidence_score + style_score) / 3, 2) if judges else 0.0
    overall = round(
        0.25 * coverage_component["coverage_score_0_to_100"]
        + 0.30 * judge_average
        + 0.25 * deterministic_score
        + 0.20 * cleanliness_score,
        2,
    )

    return {
        "section_name": section_name,
        "overall_section_generation_score_0_to_100": overall,
        "coverage_component": coverage_component,
        "judge_average_0_to_100": judge_average,
        "deterministic_gate_score_0_to_100": deterministic_score,
        "report_cleanliness_score_0_to_100": cleanliness_score,
        "missing_requirements_count_flagged": missing_count,
        "missing_requirement_ids_flagged": missing_register.get("missing_requirement_ids", []),
        "missing_data_language_hits": missing_hits,
        "approved": approval.get("approved", False) and not missing_hits,
        "policy": "Missing requirements reduce audit/readiness visibility only; they must not appear in report prose.",
    }


def save_section_iteration(
    section_name: str,
    iteration: int,
    draft: Dict[str, Any],
    claims: Dict[str, Any],
    deterministic: Dict[str, Any],
    judges: Optional[Dict[str, Any]],
    approval: Dict[str, Any],
):
    slug = SECTION_SLUGS[section_name]
    prefix = f"{slug}_iter{iteration}"
    write_text(draft.get("draft_markdown", draft.get("revised_markdown", "")), DIRS["drafts"] / f"{prefix}.md")
    write_json(draft, DIRS["drafts"] / f"{prefix}.json")
    write_json(claims, DIRS["claims"] / f"claims_{prefix}.json")
    write_json(deterministic, DIRS["gates"] / f"gates_{prefix}.json")
    if judges is not None:
        write_json(judges, DIRS["judges"] / f"judges_{prefix}.json")
    write_json(approval, DIRS["audit_logs"] / f"approval_{prefix}.json")
    if "section_generation_score" in approval:
        write_json(approval["section_generation_score"], DIRS["audit_logs"] / f"section_generation_score_{prefix}.json")


def same_issue_signature(approval: Dict[str, Any]) -> str:
    return json.dumps(approval.get("failures", approval.get("required_fixes", [])), sort_keys=True, ensure_ascii=False)[:2000]


def run_section_pipeline(section_name: str) -> Dict[str, Any]:
    print("=" * 100)
    print("SECTION:", section_name)
    print("=" * 100)

    previous_issue_signature = None
    repeated_issue_count = 0

    draft = write_section_draft(section_name)
    draft_markdown = draft.get("draft_markdown", "")
    approval = {"approved": False, "reason": "not_run"}

    for iteration in range(0, MAX_REVISION_LOOPS + 1):
        print(f"Iteration {iteration} — building claims register...")
        claims = build_claims_register(section_name, draft_markdown)

        print(f"Iteration {iteration} — deterministic gates...")
        deterministic = run_deterministic_gates(section_name, draft_markdown, claims)

        judges = None
        if deterministic["passed"]:
            print(f"Iteration {iteration} — LLM judges...")
            judges = run_llm_judges(section_name, draft_markdown, claims)
        else:
            print(f"Iteration {iteration} — deterministic gates failed, skipping LLM judges.")
            print(deterministic.get("summary", summarize_deterministic_failures(deterministic)))

        approval = composite_approval_gate(section_name, deterministic, judges)
        section_score = score_section_generation_output(section_name, draft_markdown, deterministic, judges, approval)
        approval["section_generation_score"] = section_score

        if section_score.get("missing_data_language_hits"):
            approval["approved"] = False
            approval.setdefault("failures", []).append({
                "gate": "report_cleanliness_missing_data_language",
                "required_fixes": section_score["missing_data_language_hits"],
            })

        save_section_iteration(section_name, iteration, {"section_name": section_name, "draft_markdown": draft_markdown}, claims, deterministic, judges, approval)

        print("Approval:", approval.get("approved"), approval.get("scores", approval.get("reason", "")), "| section score:", section_score["overall_section_generation_score_0_to_100"])

        if approval.get("approved"):
            slug = SECTION_SLUGS[section_name]
            approved_md_path = DIRS["approved"] / f"approved_{slug}.md"
            approved_json_path = DIRS["approved"] / f"approved_{slug}.json"
            write_text(draft_markdown, approved_md_path)
            write_json({
                "section_name": section_name,
                "status": "approved",
                "draft_markdown": draft_markdown,
                "claims_register": claims,
                "coverage_matrix_path": str(DIRS["coverage"] / f"coverage_matrix_{slug}.json"),
                "missing_requirements_path": str(DIRS["missing_requirements"] / f"missing_requirements_{slug}.json"),
                "approval": approval,
                "section_generation_score": section_score,
            }, approved_json_path)
            return {
                "section_name": section_name,
                "status": "approved",
                "approved_markdown_path": str(approved_md_path),
                "approved_json_path": str(approved_json_path),
                "iterations": iteration,
                "approval": approval,
                "section_generation_score": section_score,
            }

        sig = same_issue_signature(approval)
        if sig == previous_issue_signature:
            repeated_issue_count += 1
        else:
            repeated_issue_count = 0
        previous_issue_signature = sig

        if repeated_issue_count >= 1:
            print("Same issue repeated. Escalating to human_review.")
            break

        if iteration >= MAX_REVISION_LOOPS:
            print("Max revision loops reached. Escalating to human_review.")
            break

        print(f"Iteration {iteration} — revising minimally...")
        revised = revise_section_minimally(section_name, draft_markdown, claims, deterministic, judges, approval)
        draft_markdown = revised.get("revised_markdown", draft_markdown)
        write_json(revised, DIRS["revisions"] / f"revision_{SECTION_SLUGS[section_name]}_iter{iteration}.json")

    slug = SECTION_SLUGS[section_name]
    review_path = DIRS["approved"] / f"human_review_{slug}.md"
    write_text(draft_markdown, review_path)
    final_score = approval.get("section_generation_score", {})
    return {
        "section_name": section_name,
        "status": "human_review",
        "markdown_path": str(review_path),
        "approval": approval,
        "section_generation_score": final_score,
    }


In [24]:
# ============================================================
# CELL 19 — RUN ALL SECTIONS
# ============================================================

# To test a single section, set SECTION_TO_RUN in .env, e.g. SECTION_TO_RUN=Governance
SECTION_TO_RUN = os.getenv("SECTION_TO_RUN", "").strip()
sections_to_run = [SECTION_TO_RUN] if SECTION_TO_RUN else SECTIONS

section_results = []
for section in sections_to_run:
    if section not in SECTIONS:
        raise ValueError(f"Unknown section: {section}")
    result = run_section_pipeline(section)
    section_results.append(result)

write_json(section_results, OUTPUT_DIR / "section_generation_results.json")
display(pd.DataFrame(section_results))

SECTION: General Requirements
Iteration 0 — building claims register...
Iteration 0 — deterministic gates...
Iteration 0 — deterministic gates failed, skipping LLM judges.
- draft_structural_quality_no_templates_no_absence_language: PASS | failures=0 | warnings=0
- draft_depth_quality_no_truncation: PASS | failures=0 | warnings=0
- claims_integrity: FAIL | failures=2 | warnings=0
  failure: {'claim_id': 'GR-018', 'issue': 'claim_marked_unsupported', 'claim': 'Under orderly transition scenarios, the portfolio demonstrates adequate resilience with transition risk losses remaining within Pillar 2 capital buffer thresholds.'}
  failure: {'claim_id': 'GR-019', 'issue': 'claim_marked_unsupported', 'claim': 'Under disorderly transition scenarios, medium-term capital consumption from stranded asset impairments is material and additional capital buffers may be required post-2030 under the most adverse disorderly assumptions.'}
- factlock_numbers_entities: PASS | failures=0 | warnings=1
- refere

,section_name,status,markdown_path,approval,section_generation_score
0,General Requirements,human_review,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,"{'section_name': 'General Requirements', 'appr...","{'section_name': 'General Requirements', 'over..."
1,Governance,human_review,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,"{'section_name': 'Governance', 'approved': Fal...","{'section_name': 'Governance', 'overall_sectio..."
2,Strategy,human_review,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,"{'section_name': 'Strategy', 'approved': False...","{'section_name': 'Strategy', 'overall_section_..."
3,Risk Management,human_review,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,"{'section_name': 'Risk Management', 'approved'...","{'section_name': 'Risk Management', 'overall_s..."
4,Metrics and Targets,human_review,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,"{'section_name': 'Metrics and Targets', 'appro...","{'section_name': 'Metrics and Targets', 'overa..."


## Whole-report connectivity judge

Run this after all sections are approved. It checks consistency across sections before PDF assembly.

In [25]:
# ============================================================
# CELL 20 — WHOLE-REPORT CONNECTIVITY JUDGE
# ============================================================


def load_approved_sections() -> Dict[str, str]:
    approved = {}
    for section in SECTIONS:
        slug = SECTION_SLUGS[section]
        path = DIRS["approved"] / f"approved_{slug}.md"
        if path.exists():
            approved[section] = read_text(path)
    return approved


def run_connectivity_judge() -> Dict[str, Any]:
    approved_sections = load_approved_sections()
    if len(approved_sections) < 2:
        result = {
            "approved": False,
            "reason": "Not enough approved sections to run connectivity judge.",
            "approved_section_count": len(approved_sections),
        }
        write_json(result, DIRS["connectivity"] / "connectivity_judge_result.json")
        return result

    context = {
        "approved_sections": approved_sections,
        "checks": [
            "Terminology consistency across sections.",
            "Time horizon consistency across Strategy and Risk Management.",
            "Targets in Strategy must not contradict Metrics and Targets.",
            "Governance oversight described in Governance must align with Strategy/Risk Management references.",
            "No duplicated or contradictory claims.",
            "No missing-payload/synthetic-data limitation wording in report prose.",
        ],
    }
    system = "You are a whole-report IFRS S1/S2 connectivity judge. Return JSON only."
    user = f"""
Review the approved sections for cross-section consistency.

Return JSON with:
- approved
- connectivity_score_0_to_10
- contradictions
- terminology_issues
- target_metric_mismatches
- required_fixes
- summary

Context:
{truncate_context(context, max_chars=90000)}
""".strip()

    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["whole_report_connectivity_judge"],
        temperature=0,
        max_tokens=5000,
        response_format={"type": "json_object"},
    )
    result = parse_json_response(raw, request_label="whole_report_connectivity_judge")
    write_json(result, DIRS["connectivity"] / "connectivity_judge_result.json")
    return result

connectivity_result = run_connectivity_judge()
display(pd.DataFrame([connectivity_result]))

,approved,reason,approved_section_count
0,False,Not enough approved sections to run connectivi...,0


## Final Markdown and PDF handoff package

This notebook does not use the PDF layout guide for drafting. It creates an approved Markdown report and a handoff manifest for the separate PDF assembly stage.

In [26]:
# ============================================================
# CELL 21 — BUILD FINAL MARKDOWN + PDF HANDOFF MANIFEST
# V8 PATCH: final report cannot contain missing-data/audit wording.
# ============================================================


def assert_no_missing_data_language_in_report(markdown: str) -> None:
    hits = scan_for_missing_data_language(markdown) if "scan_for_missing_data_language" in globals() else []
    if hits:
        audit = {
            "approved": False,
            "reason": "final_report_contains_missing_data_language",
            "hits": hits,
            "policy": "Missing requirements and missing data may appear only in audit outputs, never in approved report prose.",
        }
        write_json(audit, DIRS["handoff"] / "final_report_cleanliness_failure.json")
        raise ValueError(
            "Final report blocked: missing-data/audit wording found in approved prose. "
            f"See {DIRS['handoff'] / 'final_report_cleanliness_failure.json'}"
        )


def assemble_final_markdown() -> Tuple[str, Path]:
    approved_sections = load_approved_sections()
    lines = []
    lines.append("# IFRS S1/S2 Sustainability-Related Financial Disclosures")
    lines.append("")

    for idx, section in enumerate(SECTIONS, start=1):
        if section not in approved_sections:
            continue
        lines.append(f"# {idx}. {section}")
        lines.append("")
        lines.append(approved_sections[section].strip())
        lines.append("")

    final_md = "\n".join(lines).strip() + "\n"
    assert_no_missing_data_language_in_report(final_md)
    path = DIRS["handoff"] / "approved_report_markdown.md"
    write_text(final_md, path)
    return final_md, path

final_markdown, final_markdown_path = assemble_final_markdown()

handoff_manifest = {
    "pipeline_mode": PIPELINE_MODE,
    "approved_report_markdown": str(final_markdown_path),
    "approved_sections_dir": str(DIRS["approved"]),
    "coverage_dir": str(DIRS["coverage"]),
    "missing_requirements_dir": str(DIRS["missing_requirements"]),
    "claims_registers_dir": str(DIRS["claims"]),
    "connectivity_judge_result": str(DIRS["connectivity"] / "connectivity_judge_result.json"),
    "rendering_layout_guide": str(RENDERING_DIR / "layout_style_guide.json"),
    "section_generation_results": str(OUTPUT_DIR / "section_generation_results.json"),
    "important_rule": "The PDF assembly stage may use layout_style_guide.json. Drafting agents must not use it. Missing requirements/data are audit-only and must not be printed in the report.",
}
write_json(handoff_manifest, DIRS["handoff"] / "pdf_handoff_manifest.json")

print("Final Markdown:", final_markdown_path)
print("PDF handoff manifest:", DIRS["handoff"] / "pdf_handoff_manifest.json")


Final Markdown: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\12_pdf_handoff\approved_report_markdown.md
PDF handoff manifest: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\12_pdf_handoff\pdf_handoff_manifest.json


In [27]:
# ============================================================
# CELL 22 — AUDIT SUMMARY
# V8 PATCH: includes missing flags and section-generation scores.
# ============================================================

summary = {
    "pipeline_mode": PIPELINE_MODE,
    "forbid_invention": FORBID_INVENTION,
    "allow_partial_coverage": ALLOW_PARTIAL_COVERAGE,
    "policy": "Missing requirements are audit-only. The approved report must contain no missing-data or payload-unavailable wording.",
    "sections": {},
    "outputs": {name: str(path) for name, path in DIRS.items()},
}

for section in SECTIONS:
    slug = SECTION_SLUGS[section]
    coverage = coverage_by_section.get(section, [])
    missing_register = missing_registers_by_section.get(section, {})
    missing = missing_register.get("missing_requirements", [])
    approved_path = DIRS["approved"] / f"approved_{slug}.md"
    approved_json_path = DIRS["approved"] / f"approved_{slug}.json"
    approved_json = read_json(approved_json_path, default={}) if approved_json_path.exists() else {}
    section_score = approved_json.get("section_generation_score") or approved_json.get("approval", {}).get("section_generation_score") or {}

    summary["sections"][section] = {
        "requirements_total": len(requirements_by_section.get(section, [])),
        "coverage_counts": dict(Counter([c["coverage_status"] for c in coverage])),
        "missing_requirements_count": len(missing),
        "missing_requirement_ids": missing_register.get("missing_requirement_ids", [m.get("requirement_id") for m in missing]),
        "section_readiness_score_0_to_100": missing_register.get("section_readiness_score_0_to_100"),
        "section_generation_score": section_score,
        "approved_markdown_exists": approved_path.exists(),
        "approved_markdown_path": str(approved_path) if approved_path.exists() else None,
        "missing_requirements_path": str(DIRS["missing_requirements"] / f"missing_requirements_{slug}.json"),
    }

write_json(summary, OUTPUT_DIR / "generation_audit_summary.json")

summary_md = [
    "# Agentic IFRS Report Generation Audit Summary",
    "",
    f"- Pipeline mode: `{PIPELINE_MODE}`",
    f"- Forbid invention: `{FORBID_INVENTION}`",
    f"- Allow partial coverage: `{ALLOW_PARTIAL_COVERAGE}`",
    "",
    "## Policy",
    "",
    "The report contains only evidence-supported disclosures. Missing requirements and missing-data explanations are recorded in audit files only and must not appear in report prose.",
    "",
    "## Section summary",
    "",
]

for section, info in summary["sections"].items():
    score = info.get("section_generation_score") or {}
    summary_md.append(f"### {section}")
    summary_md.append(f"- Requirements total: {info['requirements_total']}")
    summary_md.append(f"- Coverage counts: `{info['coverage_counts']}`")
    summary_md.append(f"- Missing requirements count: {info['missing_requirements_count']}")
    summary_md.append(f"- Section readiness score: {info.get('section_readiness_score_0_to_100')}")
    summary_md.append(f"- Section generation score: {score.get('overall_section_generation_score_0_to_100') if isinstance(score, dict) else None}")
    summary_md.append(f"- Approved markdown exists: {info['approved_markdown_exists']}")
    summary_md.append("")

write_text("\n".join(summary_md), OUTPUT_DIR / "generation_audit_summary.md")
print("Saved audit summary:", OUTPUT_DIR / "generation_audit_summary.md")
display(pd.DataFrame(summary["sections"]).T)


Saved audit summary: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\generation_audit_summary.md


,requirements_total,coverage_counts,missing_requirements_count,missing_requirement_ids,section_readiness_score_0_to_100,section_generation_score,approved_markdown_exists,approved_markdown_path,missing_requirements_path
General Requirements,108,"{'partially_covered': 45, 'not_available_in_pa...",10,"[IFRS_S1_5_C01, IFRS_S1_7_C01, IFRS_S1_60_C01,...",69.91,{},False,None,C:\Users\BV426BP\Documents\IFRS Data\Reporting...
Governance,15,"{'covered': 14, 'partially_covered': 1}",0,[],96.67,{},False,None,C:\Users\BV426BP\Documents\IFRS Data\Reporting...
Strategy,70,"{'covered': 54, 'partially_covered': 16}",0,[],88.57,{},False,None,C:\Users\BV426BP\Documents\IFRS Data\Reporting...
Risk Management,17,"{'covered': 10, 'partially_covered': 7}",0,[],79.41,{},False,None,C:\Users\BV426BP\Documents\IFRS Data\Reporting...
Metrics and Targets,151,"{'partially_covered': 10, 'covered': 125, 'not...",16,"[IFRS_S2_B47_C01, IFRS_S2_B53_C01, IFRS_S2_B60...",86.09,{},False,None,C:\Users\BV426BP\Documents\IFRS Data\Reporting...
